# CFPB Seed v05.2 blind semantic judge: DeepSeek model ablation attempt01

This notebook tests the exact pinned model
`deepseek/deepseek-v4-flash-0731` through OpenRouter. It is a clean
model-only comparison with Qwen v02 attempt01: the conservative v02
rubric, immutable 20 cases, evidence-reference schema, validator, oracle,
and bounded retry policy stay fixed.

The DeepSeek-specific runner uses official non-thinking chat sampling
(`temperature=1.0`, `top_p=1.0`) and does not send Qwen-only `top_k`,
`min_p`, presence-penalty, repetition-penalty, or seed parameters. Before
any paid call, it selects one live endpoint from OpenRouter's public ZDR
inventory, pins its exact provider tag, and hash-binds the snapshot.

This is development evidence only. The GPT-5.6-SOL oracle is not human
gold, all source rows remain pipeline-smoke-only, `privacy_verified=false`,
and `benchmark_eligible=false`. This notebook cannot freeze a judge.


## 0. Mount Drive and bind the immutable development inputs


In [ ]:
from pathlib import Path
import json, os, sys, time

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    DRIVE_MOUNT = Path("/content/drive")
    MY_DRIVE = DRIVE_MOUNT / "MyDrive"
    if MY_DRIVE.is_dir():
        print(f"Reusing mounted Google Drive: {MY_DRIVE}")
    else:
        mount_error = None
        for mount_attempt in range(1, 3):
            try:
                drive.mount(str(DRIVE_MOUNT), timeout_ms=180000)
                mount_error = None
                break
            except Exception as exc:
                mount_error = exc
                print(
                    f"Google Drive mount attempt {mount_attempt}/2 failed: "
                    f"{type(exc).__name__}: {exc}"
                )
                if mount_attempt < 2:
                    time.sleep(3)
        if mount_error is not None or not MY_DRIVE.is_dir():
            raise RuntimeError(
                "Google Drive authentication did not reach the Colab runtime. "
                "No project file was accessed. In VS Code, disconnect the Colab "
                "runtime, reconnect it, confirm the browser is signed into the "
                "Google account that owns MyDrive/FinDisputeEval, and rerun this "
                "cell. If Drive is already mounted in another notebook, close that "
                "session first."
            ) from mount_error
    PROJECT_ROOT = MY_DRIVE / "FinDisputeEval"
    if not PROJECT_ROOT.is_dir():
        raise FileNotFoundError(
            f"Drive mounted, but the project directory is missing: {PROJECT_ROOT}"
        )
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (p for p in (here, *here.parents) if (p / "WORK_PROGRESS.md").exists()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Open the FinDisputeEval repository in VS Code")

os.environ["FINDISPUTEEVAL_PROJECT_ROOT"] = str(PROJECT_ROOT)
SMOKE_ROOT = PROJECT_ROOT / "outputs/generation/smoke_only/cfpb_seed_v052_pipeline_override"
SOURCE_RUN = SMOKE_ROOT / "run_20260722T135306Z"
REVALIDATION_ROOT = SMOKE_ROOT / "revalidations/validator_v02/source_run_20260722T135306Z"
JUDGE_ROOT = REVALIDATION_ROOT / "judges/openrouter_deepseek_v4_flash_0731_v02_attempt01"
# This independent root preserves all Qwen and NVIDIA/Mistral attempts
# as immutable lineage; no existing cache or report is reused.
CONTRACT_ROOT = JUDGE_ROOT / "contract_smoke_2"
CALIBRATION_ROOT = JUDGE_ROOT / "calibration_20"
SNAPSHOT_ROOT = CALIBRATION_ROOT / "source_snapshot"
ENDPOINT_PREFLIGHT = JUDGE_ROOT / "zdr_endpoint_preflight.json"
RAW = SOURCE_RUN / "raw/data_designer/dataset/parquet-files/batch_00000.parquet"
PREPARED = SOURCE_RUN / "prepared_inputs/nemo_seed_v052_pipeline_smoke_20.jsonl"
INPUT_MANIFEST = SOURCE_RUN / "prepared_inputs/pipeline_smoke_input_manifest.json"
PRIVACY_DISPOSITION = (
    PROJECT_ROOT / "dataset/curated/annotations/cfpb_seed_v05_audit"
    / "run_20260713T145423Z/privacy_qa/full_v052_v02"
    / "pipeline_privacy_disposition_v01.json"
)
ORACLE = REVALIDATION_ROOT / "oracle/gpt56sol_adjudication_20_v02.csv"
QWEN_ATTEMPT01_METRICS = (
    REVALIDATION_ROOT
    / "judges/openrouter_qwen3_5_35b_a3b_v02_attempt01"
    / "calibration_20/calibration_metrics_v02.json"
)
FROZEN_SOURCE_RUN = SMOKE_ROOT / "frozen_manifests/run_20260722T135306Z_files_v01.json"
for path in (RAW, PREPARED, INPUT_MANIFEST, PRIVACY_DISPOSITION, ORACLE, FROZEN_SOURCE_RUN):
    if not path.is_file():
        raise FileNotFoundError(path)
print({"project_root": str(PROJECT_ROOT), "source_run": str(SOURCE_RUN)})


## 1. Install the small judge/evaluation environment


In [ ]:
import importlib.metadata
import subprocess

REQUIRED = [
    "openai>=1.109,<3",
    "pydantic>=2.10,<3",
    "pandas>=2.2,<3",
    "pyarrow>=18,<23",
    "scikit-learn>=1.5,<2",
    "pysbd==0.3.4",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED])
print({name: importlib.metadata.version(name) for name in (
    "openai", "pydantic", "pandas", "pyarrow", "scikit-learn", "pysbd"
)})


## 2. Materialize and hash-check the embedded judge source snapshot


In [ ]:
import base64, gzip, hashlib, importlib

embedded = json.loads('{"cfpb_v052_pipeline_smoke_v02_common.py": {"payload": "H4sIAAAAAAAC/+1ZW2/cxhV+318xJfpAJitCNpIg3WKLOLYEJIgviNy+qCoxSw53x+LNnKGktar/3u/MDK/LlRyh6FMFw7vknDn38805s57nXex4LZIly+Sm5vX+JJW10iwuC13zWCuWljV7ff7hZ3YhRMJuTr8PXzKVl9eC3fBMJlxj/eb0ZbhYfNwJlpdJkwmWCLATNdci24NzLsEoEVrUuSyk0jJmWtxplnMd72SxZbpkO7ndnVS1iKWSZbGoannD4z3jRcJkocW2lnrP4p2Ir1XIoEzOC+KTyiIBB8VgBasFGCgB8oRx1SooksWHfWLJwb+sE8VUU1WZBNkGItiuATes3Uhxy2APZ0pU3KkvwKYhLuxTk2xFuPA8b7FI6zJnUZQ2uqlFFDGZV2WtoW5Raq5hgVos3LsdVzu4o338pGCe2V5xTQvt3g94tAt6X5FX3PtXxX7JfpPwHs86pkWTV3sysqjaVxV8hRf4VyVOQGu2o/iZK/G2RHCW7HVZpHL7RsZ6yc6lyJABKX1EXVQXi8XF2dtX7z7+8jr6/ezVxft30ev3b84u2JqB+RdRKKH9BcPfvfmf/jze6F1Zyy/GBZGCK0QU73ixFYm37MnijMs8gpmzqykSr+GZ2d6oOQoKOkKpIwRbSS1vRCQLZGpu5W4bmFDEYrxFNWkqY4n0iLZ12Zi8GVJkfCOyfilCDaQZHDSiEVsoVtZRjXTV0I0MGRI0BaUWnC2SqKpRDVCyKsFm/whVLJIGwZ3Ve0iqYlHwWpYRaonLzJE9LALECpooxS50DY1NjP0u2sHK0CFv6RXVqMiY3nGNlP8kqMhzqVQlsgxJjvRvCnFX4T2eUBKJgEY2OVRocp+YGSbGRXKLjOjTyUdh13ztIRobmXi9ameO1QXy1B/o6bRTZVPHYtUm+qU3iBHztqIwcJLQgwkUfUGtCe/KbCc8WTGlayhj8tkH1ESZKLZ6t34RWBGa1yACnLB/s3dlITraRKS8yfSaXi7ZVqxP7Q5RJF9N34XAQdOvQIscyTZjK7z4Htws7MDhxpknBl6APA7ZEgeFBoE56zzA6vK2j4MCKkcyecr0WnDAjloBjJW+BO3V1JaIqq6s92uicNa7iLldwwB+zfbWgEFMeRyLSlPobObZbwS7LozGB9G1JL93u4yfTOCzfET3tN0orrzSVLBUWziVChGpHX/5/Q/jnYBiCCvWtfevy9OTv/CT9Or+h+8e/uwFg2wveC7sti4d6MM5mKxALBodGxobnp8moOp7LhKO8U8mZ3IB1Eyc09Lu3IosLaosEcqPM7WciWPATv7WP6064GiK66K8LaCjMuDhE1y77djDZrE96LbLtOXQszR2cgkI+QeORHFW1zAo9f7uJHWZa6Uwo/aK3Ts+D17PvRY4Nos5zfo6eo/+IxO/l7fzBfSWU+sgTkxLwDdoOniCrJCxOQKoSP7KKmEAhB5sgwDAYKbkiPUziohi7I6lQX46MX0yA6aoQLoFW00OU6OZuqCtj9UGMZtwQc8VPauqY7QJNr2ezaI/I3uIsK5zWnZgdYyAFzzbK6kigvDJmvNhPYjIeKUvMrt0pNLmXEVOnXXA/+vxaD22boBjRK0jVwUxgDVDSw/qLDVGExauhgrTSjgqGvan9aBGnrTEQgCVL7s3zFyhPjCJFl7LLGtL3Bt5isrc0B/UHJ223eog0ebe2zR7vo4Zx6zSThU4iG7QvFFz10Pcuemjz2UmZjCOpgN7khiIkl9EtNlriiA1JG2R9r3KM441wsU/3N9Ypd/yQqZCzXU3uVuKbkQ9wbmb0xcWzOiLtbKpq1IN+z6Z540mQI8qWbkTm+bNDkoMB3/Q2T+5wdAGgzYzqpuihZfp67HfYxSCHh7r9DJFxNqa72N41a2hOJpCHwTKHSMaEyKGmycD2kEjXE/1ZwMckQDfKkkToyk9EFnfJ3ILz4OZGzpDu8m3HG+l3pm8CkvUDXqRjRfQvIgBK8kGtUsd5yYr42toxigufsbzTcJXjhIVwhP/xenL79g3jD6CJdt4XjAuFqtL2FSEnb7h156jBm3c+k7c2W9+4AylCTlSPBU+jd7wASZgYyY+rQjUOMq5AKggvJZqCX6xHqjgpNzDN/612AerAV/YlAfGTKwsycScTDWMQnpSfvBwXFJRhRi165rvD+VdzkkZC9AlZY4fXB2X4BPFkukGGBs8R8jVo9qb81nGh4x7D7iUmeVQJeFHmSNkPK+O8lClHcj9UdDNootzVvIkcjcy04w2tUURpWN0SZG/coepGePvkOImke1TmJXA8V7llgZIE+Ii53MjtPeYEwGE1ocE43AhDKQUj9xeo1yAuEWkkY8LDsx1a8+p7vWBdHLB4d4Lib0ZU5Es9N176FWYVCJQqqRjbO01Oj35cb4qp6qH5D/lE+BZ5ekbybYbSR96E8KBsmqzbeYE+6WoGtzFNcCNjWC/ovn4jY7DD9Z2NAyk54PXVuctbuJEZIwbBG3pDpO5uE26A2M2PAsfhvk1LnJ8+6DWH2vKLnEHHlF5bR7ngesWbp36DIktbsngtffP4jiuuQgbbUeuddBmzPONfxPcsSl/nCUkV9GVH1exlOtznimoTI1TBCixJgTsW2Z0cB6Decp6LDKFP0G1SbM4W3Gm9z7MYMK2pwDAPFHDZE54kPmH7FOkkg7MZStSX6qC25U56OnvW6iXhQKWsk2zxaAHM5cyRxgYpyRgMUhloh8xGGhq6Q89Mc1n76y9u+Iml80Gb4RBR9xmJVwNwckOilGsbkap/g01/p8bXEF2bfCKbcoyM/Hss78bYV1o0xo3CIRcDl5avji76C50bTZdC1FF7exV8DbFJul+ouR21MXNSYWoe3uMmcw3cEWWGj2Ow9kwZPTqcn6UujIYPE7tCVjR7rAqq/lhzOwjgBx078GB7Plx7Q8Jn2XxhHTAxLp3ZWhvgNpJ0LfcR0PHQUaM1QGPIwPUmAvo3AQxD1MzAP4GXQLdewh3sdHfZ9wP2I3mPbC9HKyRN/E4LBIiGZZCO01Gn9y95sGR3Wfg9ArUpX+3c/UY7bGkPegWBnnacsbeKb+viFy7e+j4Xtev9343cHfq9IGYyhhGo5N1OaUiZ7TvhsHpdrgIIVh25pjGpJ/t/lvNU/tj0x79LxLIEZhfnT4vplgvSaPPoWslztvJJQhxw8IRDh7id6yIUi34X7dQqsn9F1/RNs26BcB92E/istCfInswOnjMbfF/AGZVvsh0HQAA", "sha256": "a35e8fdfcaa7e486f7b28d67d587f4cca8ff76fa5c52eb6ba8806f44804d708e"}, "cfpb_v052_semantic_judge_v02.md": {"payload": "H4sIAAAAAAAC/5VXTW/cRhK9z69oyLfFSFoEyMU3raUNvEgcI1EOgWGQPeziTFvNbro/Rpns7n/Pq2p+zMgG1gsYFodkVVe9evWq+Eq9+ef7f6jj37+/+U4lGrTPtlOfitkTbn632fweitKRlPbKekMj4T+f11c/F+1sPk0mfYhKq3Ty+UD8tLde+85qd21sGkumjcGPsC90ox6O2hWdSQXvTgoGKpVxdJaMoqPFKR2p4m1ON+oRD7swjE5bnG0Totlo52ivsw1+q3zI6kjR9mzMt53KseTDzRr/ziF8lYOCneq0N9bw2QjQWL9PW0V/jNRl2BvqbIJb3AtRd46U0zty+AmrTUCgsWaLwO6DnG19j5v5oDOyj+FZpUMozqhRpwQnqtfWqT6GQSEdxblljjUivl8ol+hxuu4yYAie1L9+/fmdCrtPCOf1ZtO27acU/ObfG6Wu5tiuXqsr3XU0ZvUfFYlflYujpeerLb8ZScMq4cUPV39SDBzGELiQzoVnpFlfAKyG0tVHsZlhbyL1X7HE3cYaxWWjBMOR0ZasFkOp19XHzX857M3mYbp/DVOKUtFYHCWkdY36Uzyp9uLQVgEa3B1KympH8yEzOnKYXnnS1pDaG7h7R/CHGAXOwFcd8gcNAM3Cp0x/ZH75t0Q4w2U7OslrCu7tfVLPB/I4g+mmo2WIPJFJ6+splLhQE77aH5u/tWtR00KX9odXr161ah9DEZJVE6GRau/rM/IUNbNudVfr2iKsz8VG+KNhRIO1U0Xban8JmxhWHpwZgo4ORllY9aLu7OTi+YIRHLI77rlYvBdQU3BHOGR8drp74j6yw1Cy3gEQqc4MCyOsdJ9hBqJwk4Gu4Pn9xNwzAsyZvkYPSU1DZCQWaamRnqf2+jLk/2XD3QCbL2UFCgLki/UEUlmfSt/bzrKusX4NO7svoaSqZmiMSM5KpjXg26nh5mZcZKCAVfOxagCn4H1Hneb7rG+dTnw2PBrLJ4JRMxkXDfxae8IJbp6pzZFD9fRcH7OP90LiqqIDSyMgOktr8pZIDvxRCCqyiB4xlkvE/BwoaxRMV0GtvJ2EVL2RfpA8BvVs82HlNfJhNq1cfiHfOPEHfobypLKHbvJ5iTmE0zNzh8NGGbLNhZ/hN3zvDzx0guQAQe8OGmR0W6lHxiCQDo+GGyo9oW216HCIg1COa8yNS2Zb1eBzYUw6p3lMdPUdJMTBS1WlwMUDucTxnOA1UcwcQ/Er08YYOryyVWNwtjttVbT7Q8YZJUMyiMPDcYa0wcAhzn2q2kSaiRZMrX3Uw6DjFkh42qodMyefJNyDjoNEAsz1eIggDnrobuLGC7bP3YQ6hWj/lNSalFGKhkHbk0ETrNVh0JkT9ZkInsw0YSi8FYTIqEzuGECLOlo23groe19vI1DEh0DsKJ6tFyc5ap9QVjs1IjC3Q5NP47fFU+MA1aaVQbHlFtTpQAhuHpTlaDFU5tUDup+4W6MuCKonmu/stNNMQ20+YZ4MOERi9sFfG7Q0zx+4m14ufs34ixR6XOM0QbWkb0pDKxARpAd1q5X8EZRw6NI8S2KA0prb4vkP2EXy9Fa2HhLkQdPCR91ip5ouhW149bZzIS1MF7FmV/OlpMCLDoqXm0Q+oZxHas7apdkXaDXAYo1NySJW5IImhqprTkdE9MQqsYtBm9o2yAGQ1tHHzGUE+bqb9jKEU/el5cyLFmUZQd/Al9cRaxN7t1mWNJV0T0vzX0/NP7fxheLmg52k2jAtYA6Hs7LI2vpNwjJHg1gGDmQJGd7Ogp6wXLW1WUrJdFhmvWWBw1hDg7C8c12Mlv0yL8NOhstAmsW3L06tu/Ej54QcOopQyjOvOZHrqz5PSiKz3UDnmFULGw1LpJNgZR1Zo2zgtod08TRl23m7W6ZkXV8W9uKI2QInXCp/3WUm4ZBXMPp7kAxLLrohjqzUAjzLXQ1uxlDW9CbERgQUPcUqccG+KsDcSXWlF5Y5qn0MBcSoCtgV5SHP1MIUjLQvrj5g/FxJMxMN9aJjIKEcKSCCTBB+mw5IfHdaM5MQz3S/ge4brJJNlf2LOKWv0wuqoS1SKgR195SfQ3zaSq0js63K0ORRFiJ8fji7tzsrgtZFy9izE1azedJMc2reoqdRwzEYHXkysWDh4f+bV0cGTe6+LgEoITpBqpDwgcSzcxbmrXTGkef5/CFGqYPm1utIfISwJNtB/prQFS7fKg8R5QTt+Te8TcOV1VAfwV+ZydP0hniZupyv7WACcL9IFx1yM28bsimIXlR9uJz9rE0v5WVRjTih6FktvsAsdQTBsqGpXfbVMVCXtIsxAM8hbmuaXHnAdRQun00bfjaC9Sh3B63yAYJwWoHbLmsQ1tSqvoGHdWcjXkiVHHrHFasfS2vh1U/WMwjB88ATDLRk7NGv3BRLnLKhrksRFo/30R51d5JuT/gmI9k4HrF+ZNV++OXh/u7N48P9xxbfPB/ev33bvLl7d//2/u7xgW/J58qHu59+/u3d48eWa2sFohoAvuk7OgRnENTFtrR+vQ3oJK42cEliwztq/USBSuLfaEfibes6DeEJewNgdWFkvNZVlBfsqptVTOqwwKCtqTGG+LCIDKAQcwerA7azJ6VH1ATfMjebvwD/EU8ULREAAA==", "sha256": "1fb776f95da7624d52d6ca659be102fc6247e5f10955f77b074b000b12084414"}, "evaluate_cfpb_v052_semantic_judge_v01.py": {"payload": "H4sIAAAAAAAC/7VY308jNxB+z1/hbl82VbLhQNAqUipxkDuBICAu3EPRyXJ2J8HH7nprewO04n/v2N4fzoZwd62aB5TY45nxNzPfjAmCYLpmack0kEXK84R8LZMVDEWePhOGP2ORLXgOyRCleMK0kCSBNaSiyCDXhK0kgPkWBUHQ6y2lyAily1KXEiglPCuERKk8F5ppLnLV69VrclUwqaD+/VWJvP4ulNOE9kDzDGo99e8BMX//Ejk4uYLp+5QvarFr/Ok29HPB81W9fpw/93paPo97BD9WQMWSF1pFK8hBWg+jHDJB0RKjCSi+wvUoXhYLut473KcFLwBhAqoy8QC4tk8RoUzktY3Q6jafVLCECsniFGis1oPNDQUZyzWPqcHbAKhaAXXP9g+P6JKn4Bb7P+pwFSw0/Ibn2y5DlQqUrRjPla7cr52ApxgKTS5FUqYwE/qDKPNkKqWQY0J+JqdcQqwJPEFcGscI5gpkC0gSSMiJSNmCyDI3kYva+/wItMYIRhTGBG8pJNzlYighgeWX/xn0H4PzO9z8BtC9Hr2+uTqfnszpzdXVnE5nn8kEqyKCfM0lhnwFOgw+nM1Ozz5d386n08/HFxsngn7P/4mHXZBNZYRbuvuRBCXSNYTuznxJtmTsBqQKKh0WKEq9oxGWs4H07uBLr9/bNOspswsjEohSF6VWozaRRw5KQz0jC7QCSDpoizVIyRMIajUSquAYchk1JGUCMlKilDFQTDq6v7d/tPfr/v783cHhwd7RHwG6eHJ8cfb+5nh+djXbRMn+QN2WCtVIFJBLdBck/fMR8gN6SA8OF5QdmFR4R5nWkBV672AUo/WFuwsaNCZOpx+Oby/m9Pz29OPldDb/hDa2zKKl7cSMDB+mQaPh6ub45GKKx+sjLmNGq0IfHmEEKEvwLI9r68a1CCug1XBydfn+bDY93eFCnZLJqAWUSjApbcC0/nju3M4x8Xao8nHIQEseK+uOU9HrYSm0FWBBdrj/4vK/wYAaXh/bhHM7VWF3l+seVbu7dcym2sZynwx/JwiXvlNaDkxn+DLeNI5328EZ4aaDfc+1+lDLQKHn84BI+LNEljTEVqSgYTKXJTRFp7CqG9198tPErjgF/XFDHpJxLMPPiB9Y8g2Dc4OhbdeVG7GpE4ZrCV8uQQbORIEshFe2QDGMhbnj3y0BmmrjyXjDjaiS7JMlsnklMmhQIjxvEYs4FoKqOOSla9EU0A6D7ZL5BCw2bSYYk6Dm3SQYbIpI+Iqtxoi4b69JrDk8Ogn7rd1/uWtul0DMFaZpy8z/6po2hy1x4QV3cHvbZbfCMKjCNtjCa9Jd8DpTJ+vRsqmvyOSfCl8rCRPLhGp40iHksUhwMpoEpV4Ofwv6HZVV1aLOjiLXeaqcrqSCJoFxyiOIJ16a5TGEXXUDW3F9Mxd0t5xeKR5Rm0n8FPLvSfyTSg1pp9MKjZTFD4rUdVbXxabHDXBtAgY144E0eWGTcO+dlzxBUcpCKDA7dmCmaZo5EqPeYExf4UBfS4yxMFEtdYya6rk2ysVjWI+2Ee71I64EpmTGdNj3jlugxj5K3mabjY3lsZeivhd1FNq+2Z7Yil57juemdQdbldsSo5umUMIbq7q82anYKqlePemRaPdY9w5e23pV1WuV4el88UMtUh4/b19zV5wNuihtOL3j5H2JTYSuRJrg/geGY1RXAFLsGqXesV1Ivmbxs0lLvuSwS8sC6/o+Y/KB4rS04osUtgX9C2K9GolwU8t5+wKseQAn/7RMsJ1gLcmM51xhR/SKbomFgHSiItJUZLCptNaUIeXh25Doe8BTTyi4S2eRlsqKXVxcNuwbdfW+F/reljkOn6RiXPLxej48jI6Gn64uNl+r3ow0sGxlA0NMYKJWcZUOL93hoZpwo+wh4TKsxl3bwgeIEDdU/+B1dP/ko8Su4bi3MWPpOimzQoUuF1FLbsChTMWcT1zgsPckaGeyPyDK5OsDPDubXtJ26dx/wEjAt3hecV01fNmHN8UXOLYxMwjVb/FoxjJQBYvBka5dlEiQjcCxXJUGymu7E+KD075HEc8JpYmI8UXgnYxYkhgz9kgYDIcNAwQD+0Sa2GkMY7RkZaonW+Pym8ocKbytyY3Nb6pp/sPhIHpbXz1Ev+2YDfw3HLMD9EaIKm1+cKp4ZZjWLlIzbAwuNkYAI7MhvdHUXpuwt8friTkavfIe9ijXyfhP1F1Tt5Ps7Hg624qodNoFP12R6RDE/1oaBjjziqU5JjT+H2oyIQGlBkZKA4efw7T3DwHtsvABEwAA", "sha256": "cb2b1940a5643f4f44824a5041081e75ca20177b2f1c2fad67e3af45900b0715"}, "prepare_cfpb_seed_v052_pipeline_smoke_v02.py": {"payload": "H4sIAAAAAAAC/7U87XLbSHL/+RQ4XF0FsEiIku3bLG+5jmJLjnJeWbHlreQoFgokhhJOIMAFQMlararyGnm9PEm6e74HICn5sq4tLQHM9PT09PT3jO/75xVbJRXzEi9lDauWWZHVTTb33p6c/6v3mbHUux2+jg69VbZieVawQVnk9169LG+YVyfLVc6iXu/iGh7KdTVnXsVyltTMu05qLyk89nWVZ/Os8dZFxW4zdgcAV1V2m8zvvTSrV2WdNVlZRJ53cZ3VvRXHpvKaa1axRVkhwMW6ZrUHD8sk9+YAvkqKOet7t6zKFhl8gsYmMBg39bKm7sGAKaOmFVuWt6JlWWVXWQGgiqSqkia7ZYDlnFWrpk89a1aktUfTTHopW7CiZoOsGKRs1VwDpDSZNzCLq6pcF2lWXMneXlMS/CtWMIQLiKyqElGoop7v+73eoiqXXhwv1s26YnHsZctVWTUwaFE21KHu9eS76goIUTP5DOS8zrOZfPx7XRbyd1nLXxXjQ8zLPGdzAijHeAvIwvL2YZUXyTpv0mze8MZp0rAmWzLZUj73Pfz7a1kIoKukQQxks3N45B+a+xVSQbw/Ku7VJFZATmSD2lulvV4vPv/08d+P317Enz5+vIiPz372xoB8xIrbrAIWuGJN4J+cnr07/Xz+5eL4+OejD1YPP+yZj9A56HnwDzEJWrDDqGJ1md+yIKRW2cJrtaEPLAdu5TDiRZbDuhhdI+TGoqknL6e9sPf5+PidHNtCZd/zgWzA9s3+fA1rz9L9GrZOvCrLvN6fL1azGPlz3TD+HrbUoQ+zOf356O1/ObMx4NKLDtgGw3DgEmicrNOs8WW/al3Eh8PDPw+/O3h5cfDq9avDl3/bF7sv/iXZX6zznHCBP4APTPCnj3893jTDct3ADOp9zd/7JAZi3Co2HoexFBcxbLuqgj3g994dnxx9+XARExVPz2CNYRBNUhjBgaEHirMCho6Q63MN6OTLhw8EbTscXMNf1qxxMPjp6Oz05PhzCwk9/jIpsgWr+bi6N/Lnx8+nF6cfz4hKxiJCdzVxSWdDMAHYAwfYxy8XQAiA+Qnx0OTH1WNzgIVLsy/EYsrpUIMo+aOHMndWNg3swIM/eeWCZM/L/nD4yoPxmgEXVCiFhHyqQS4UTZKBUEgabwltvINDAFSUxWCVJ3N2XeYpil5YU2gTlLOaVbcg6ZCFBsPvBoeHIcjpT+Vd7c1YXt7BiABohmIwqUBYgvj5Iw6Wr1PoNePim3QEyIe/EH66MQhaoExTZbM1l9kk6udlxfsC+EgR6afTs/j9p49fQDScvY8vgEhnn4FcgP35h6O3x//28cO740/x+dHFxfEnXJOKRfNyuYLNHFT+5SR4Mzo/PY3fHkH/d0cXx799On539Pbi+N1vk6PB3+LpXng5BeFCcDdBwZbJ4Nfh4PvpHsCb/NP//vf/DKbm2/AFwOidn54ffzg9O45hAM4gmiP4bklymFus1aHf+3T8H19OAaf45MPRe5zYA21gn7RHDcRJ8nFTrZnf5+9noNOul0l1EwPcq2yWs/EiASkmv0vOE/oxVV8fYaj3x/8Z8/kDejjYhDrZJJvhxGBWUfynvcF071/kI/y+jPBh+nDYf7yc+ahZo9Ow34YRvPnhD5dpCLS63HtzMBlEl/X0TfgGn4M3l+nDy8fL8I18Tc/iAX6/egzeYGd/C2DqMoC/h/R3a5fL2XXTrOo3o/39y897v13O7u7uLiP4uRt/AP1934A9BT0GKtSrr5PD138mhRGgZhyRCgm9wY8eMPWI4KXZFUgPILFQ3hHvJBTSXQbWBHaNyhUrAr+a+SGqymvYGjnjEPAf7CJvlpfzGy8rwKphVZAny1majERLUFZJGhwMD195Lzz8X9j3Zr4faggal2i9QvUeELxQTBqMkUJ+v2Zf+S9A0ppow742wW2Sr9kIJ2hPVMBwpkmtQbvPy5QF/rpZDP7ZD8OOIfIySWOS7S4pc5AQEzRVJjBWH22L6XTURT0aBaTMWI7TRUmB5gSHinDQOsB9GRKJ8RdSmHdCcwHfRCihVkEol/2ughVo4QpcBEJx1IkuzeMMrCiOB6HMjYpoeZNmVSAsjPEF7PA+iE+AEZc39NjNKHfAte58+17B7hDhsX9ZbOYjQBPnSNha7CFYiaYXEH3S9XJVB9ASB6vRXk3qeZaNT1CW9MHar5r4ht1zvENvz6OBBZmE/RRLAzxG9AFY2UiCFcmSc1IfeXo58myyEdWwpWSwnBvqY+wSYA9uLiJcn0ML0WwjPRb4l5fw0t/3Q0VztK9wfFCsEliHhYgt9FvQjA2Si1a6770w1q42tleVZGBB/ozcflxVZRUs/HeGMyJp4LF6nqzABwHPxLsrq5t6BbiOvAeJz6Nv7UgcTZCTBPm9tqkMkyIQkkbbGAZb0jeyZ6QpY30lIjdrcOEcpu2DvR69A6PzpALCTkfuGEBMYw+5g5NA4gKjtS35DMEpZORDSVXH1ZatJ9GqBHh1DSpy5NHu0G1hPuskj7kOjVesQu+QGnL+1C2FPxorv1HDdJu29WpHI+6GKutufs3mN3EN1vi6htb+PC8BeIx7FuxloYLnZMQK9fxIf5dZvUwa6G0TAXbUyDMpSmwOb0GokzwNrf0MH8R75FNJ1gj3Rx3opsDZXSC9P4x5ZwMvaKpR28rj52K5LMZYleDt3ysQwN0amuRvGAKcmdYkYaV+WWdo5HJHI17kyVUN+3gyDQlV20zagls3ahK8R3DJ3gTcanRdgZLrQlIPhRhJDbHCJDXGrS0mBAHnZ7nFxy06+/IbTOThUW5wMVGjn7kPQABlaVbGKCDuwcSgKEMtmR38gWhe3/oGTxbsiiRIjAZ+VeZbms6vwf5kxRWDNjXEAequRrNkfrNeKaFhMa5YPYk6rUxrQs9eHEWJrLgFqpbgIWQgKQu0x3KIS7WWyCQ7rUdtqhCUb1OkKscZtwqqCK5raKuIrnKraISFsujWYCZP9A2Q1lZDrUEcQrrcAq4pcgKG41nZnKA/xAmDXSw4rolJpNaqj3/2O6Hv3qiK4Gi0mfsVJyV3apvIE/yMlOXaiYgr+BNfplz2AzsFmwA8lb2nmh9TiDCxMS6sISfZKhahrLhIxo6UdvTOoM6uBBPzickN82Skd+2w3xVbMk8T8In73qIinpMbIUZLjkwUBSzwJUVB4siffe/719+FfaORnJAnJlSTCSmo0geXfyiaG/yFNjEYoIQDMaOFxk4+/KL2r/dA03kke3SOkUlgPA3aZD+xn+jDRKr9VZbFyEYgKfxpxH6B6ZR+GIFkC3ZvBzl2WhJk6OTNIRB8T/EJMA3WYEfAF3O3bMRHGhc80MJxcSIAz0UL9RKOgHYIxsrNsPsuurTMoSn6L2Cm3YHXGBKheCjguUihIK4gOtNATHyeJ9nS4yN6fMSNeHVYVL8HSivMW0AkSo7mrREgF08JRE/uf2UV7txFdhUL+W5pdwi2uc1wY/NIHdnQ8wRjkEI1bYLhtLJB4Camfrhbu5Dqd41i775/WLM449p2fBdSu215rbAtYB1z2Q3LdlVAwaG/Z2jBtiujLEoHeaEiKSVlQSSVibKrNdZWm0VkrOQonqAf2Ci4JhTVlOkfI2emDCiFpqLI74Qgkt22of4h/FyWsPAzFqaLdcJvwFfR17JKTPSawCUhQChQBVHsGNXY4ID7Cy8Ph6++AYc0S4loc8xckHp42QdIKrEpB3LCZ0pZaK0rI2hNAoIPgyUBOKBpuaTUxAgnw21IEQggy/V5ITbtDPoPBuzH3x4UXPhNMB99NxTH1XtXyK/BrAtGjmOe4eXjbI5xiUBD9ivjs+LN3bn2doT0hIA7GHo/jAkY/v9w+CRfwsxH877LdY0aAf5r7hgrECzmHA6HmpvQ4sBJhd4PHPlt7u5HzAc/qC6PntRoRBdS28ltkuW41mKIGYQHWJ4Lx4QHWmjS8g8EWrrIgfa1kaYNsI22BDtjeDiOSiHKfxgqg2bSu+amSg3ZmAajYyB0+5ubQ+4hXc+f0BBcaMxMOM20QSCIMME/0yhZQQgzxd68Bb5FIxzDiSwN3F6mCX0/5nFvj9O0e1cBHvgZ8fnNj/5eZkWAzxI1TUUiGJCREBhtwBaxsklqoAFz2IIFlxJ+XxJrwvOKGdhkYeiQqWY5mcXdmwtTNFO5o27gacijwtcgeImFZfc2G8MiXoGZXFOQiZwOK3q0gQYyFoqj/TimISyyOFYHLSM4EVmxZjabCLzkohMvW5AmOIZJjxbSGPZzEbPnPB47k1ZLCebGTct6UsDtDnZjmvoeJBhN0euyqEShk0e3MwcPcG1gDsmpMpKOWdxYFZvEIpnL8UBjiisO0xXn7pgVw7WCvTzqXjRC7oKaKKRg5w1IM2EDg/kM0WMOEzVlTAsLVTWgQMe+Uo+awE11P3IESFI1mFfAJUadzmmAb4ECff0KWsAL7SjD3KHOJvgruyex3PcuwNcWP7W0dhi0xZzZgmMA+wV5CYkYknJAhH70uKKQLeCdgzySS/J0sEy+BsM+bw2YQ/lUoGDSDMMwNO1/yN+hseG5iVg9BocP/QV8MIAiAh8gfHrELyFPW9ELBChgR7Q4mCUkFEJTs/KFdtNh2K4vhAofm8Qe77lk1ZUSTPSngy2M5QREnFEwoExATKqOxbvJ4GA6OZjaBLY+wUhIYusdp6vmCpBrXRDkGk00flNptIkarjHNfsNEwLdlUIGVisFNe1b3lz8nI+o8xUTYxKo2mPrwTjVDdpqaYkV+6RMviqHMxKhyAciKj6lmB8UFlwDqsZX3kQpf9ndTbEIk6JdokPA5QrHaBxiagJMfQmQSJXGLRtjnlPUcUPhGFM5BlH+ZoAmbpZTxiKjqzcn4uHhNZGGRPxVyUNbrTCXz2pOM0LZFA5+KtiTkCc8FTkP6vM1oPFGz4v4495HQNZIoQakfRDkwV0DNzgU+On1huD42blZYbCJjtNOnoSPG4c5PWrJaBIJwh2/22ijUYsQxBfUcxPpYFLheFvVYK5u+KHbCxZVithXjQ1msJ4Qi35+Gz5uPivK5k8JMaNsHFbEjpRKjrC4SiBUlxT3kZGGbthqkaywyxdI42ezZBDdzQQqcJ4bgXlnL5QOt9ysUB7TQSWoMAQfow/UtZDmhze9haKd4hXaPy8UCsiwYQEmqLCmEun9hZHSz1FD8optpCuDqP9c8gLVWAuAEfBmPZzMh85KDzUQwByQJzkUM3+N4cvenThaMxChuJYGRlgAyMCjx4jqtxkRpIBpvz6m/b9Xbip2bGIhBVHiRfYVosqCRDCXLUihlPiOegh6/j0kjILpmjXr9jaYNVnZA251h0tOCxLCaI18omrZBG48KeAFel200xL2AE0J3nN6J5x+5iJAruTtme2IjAesGIh8YiDkapr1ulFCB/J/QHWgvixlx71N+41uLe6HmXBSS3AgYoQ2AUk32ezLqCGNf4C/jUi41FSzFba4PIT+ITa+t+gVLsCa8DtSWdZQzamy1Nz8Jyw28ipqnoWFFvoLIAlquF1BvAHwLgS0JtM+tP9DNBVb2X63RAar11jSKP9EX76ipJKsS4/O03mJGsICYAujuUK9nge/Biugeoq4UnDqz0FKBlgAtIWukxTWxCkzBa5xjDhiqL8jypgcjFLGlIyTDK9nPF+EC0b8bgNUZNZrobFJwd09QCTxfNHICNi1QkFUI2i/3zImGrntsrWYpy3mtVlTyfhANdRxCZvg5X4pyY1GIw2uOOzQQvjctzs22qBXLNj9sK1wSZSBQI2eBohhfvDPYyItsCsNz5oSgz8CE22qLLW+Z16vxv51xSxkYtvMf5oyfkEVZSDBWtI5H2n33LA4ErFM0saE+aYudgwdqPHWgJpmjghHnWpw8gIbXjU1HGi+U2pfCStvQ6BqHR2fhUA5Uo+u0HRyy0fAtt4HzWkyGsfIgLMfBOCvATwlMtfXNU1/zBlOkFqBWbom+6pxSq4uZ89avt83+vT4MRB2eYdy7G6RvllpsKwl0N1Xf3n+hs1khJJehct3ubbruhFiEUKUGJAReHOuSU1VfWaVOyhNp1+uJOLVZFiUi0kb39SzueC1C0na79kscWx0Bk5a31YJn5+Re3PhFlv9tLyU0p7Ke5aIosPP7MgFPAI9rYXUMcKA4/+MWEJK/omLnisIDKtDCJZkMsXSO6lphfUjqO41Do+gPwW23wF1eFg5/prwnXu+Hv6Q5pLIk8qCBYJdNZcmm8aZyDM5CUNB3i1BUMDAosQGIXDMOS5+JijFkBPhvALJJEtrBdatL52Lv6rSJA5x+U6sCS+3kByNSha4LOYuoCFc22eXHPv9oOEJ1RG9m9zpxpFr7vPqbVzGFBkuacqedvHel0jek/EU5N6i0qpzzKP6OlInwoxWF2n4gt5HjzclNJ9YpuIhzrFxPI3rLT8GOeXhAtw6NvAGPKXBvhrcwEwCWKybaikJ0R2bvdF6MDUsI6ygH+X3K+er2u6Sb3YFoW2waeJu+NHS2uZPYWwymFVqUwYGUyXA0NMBsXLq98dOiJQ7Bx3JU2xTmIMbyBLD1UXmPY4fyEwFr6rQ3Zj42H7oSozrwy2Nj4825HnU82YRpEFz4eQCiw6OU47Ta82CHCJLL5rb3LF5OdjtfUwgYdJrb21MxkqtisVMUO3ljuTTttnJGVmv5st1co6SORArnDfvRr22dJBGotXwwUqxaGElaciDCgBM1CeOOqgqzb990bPqmP6OFHkakd+aI1RQbOCKcW/EuYB72tS9FFIqvAood8UhxwAc3IjlPllJdUiJalaugtaZh2JVAafVqra7RT+4UjKRZnTavcdi5Twzp3IKgFtzoajbdZDjSKS93jnol9lxeQ5Fs6mp1/kNQG35PbfHCT3zY9b5bbOmdNvUW23qHjb3R1t5qc+uP4lBwTNHxrhasqDKwtMAGghVNanV6xmq0BHFy3T3hK9glGKRRRmFHK4iPtk13W0gLQSiozhdGfXw0l1IerbS6PnRMnWIqYrPTjvRHYme2GxtebUsrQDelQjqGsfaeLNwbWcc5Zdi9a+CN+2nEubjd58ULFW9sw3M1tNLmraNd7gxaBqKYh/u6i6u3HcVXkJyX2+Dog2mb0TYObW9rZJ/Qbp8ze9ahtO2O5ebmm7yMzT3MI/uwuaAUsXrKaI8d9o/UaFZNmBP4e+oRWbGdRF227g/x0wUcdViWXTdT8NWkkj6JS/go75ggF9usNHbhulzBw0HOlRFUqGUcGTbw7CsChNZgbpreDpgolLA0gYeQfby7wzzhBfIS1ee6mSN3i9tcoqK8C+SFLhF8CyGtWtJhRiwsMWMTFWwHRmcZN1zpYWyDzdGSkQ7TyVfTnRGUdif5paPv8yWD7OleLKI7t0J+7c7aFXCFalc9QL/3/yWOlNBQtDKaTraeS51+28lZbYPiwhgVbGa0DhLotTwChKQwrFgz1pbrg0IyZyLZv6udMGaw7cQukDMDRhKEOT8piGiomF9CYijPK4qaEwKyZbixN9W6dHc2bffujI9obJsAvtCRuL12+lS2DPXxaq7legnMCTes1GB6ApAul8vpJi9kMWmvpg7Bwi2z4fJES3nE+gAC98A1cxDIWICqLnSBC2h4rfyWC2iMGT120owmEJtKpkVB9Bfw8PIIzvGWibiTIuyyk42T0VDr8BlULavbdjNnryf4t2YBpOFA2yZ6GKVwzB+QB+9OEQlqbobR8ACK7KLha/z7GqqUZJFB2DqtbZLGtZs6GErGKGJe+8A51zAsNrA67yziCaKMw+y2KfziQoAykZiXibTMne41VmxRKwtWOUktUYBfIPhF++UtHshIruAoHNVlMKzXhIvcGCAGES0ueP4iKkKsq4VAQGWg5ljUkU5wmY3WxGY4HjEXF6jxulTpEHVKI7WioZtfbTtj3za87c99CxL8nH6bm55q6u40c59o4j7ZvH2Gafs8s/ZbTVqlcCF02fBLnbomaW1mXvbdJjoWLo5MI5ZqGJ3xtqtOvme6DBIDbKs9aGkIaDasdoZHuwEkE9pPliJ/tDJQpnnMr47hfqVqblwkIxt33yaD/i9UWh227pXZeEjaOiAtK/RN69rCTj/KCge8WzGGSxZhh2CqX163GJ2p81ji6hh4WYENrhocVVdrDEuc05eAi/sVMsw4jtNyDtcHGj2jJE1xGOoS+IMB2jEDMi+xNgZPiovKAn7eZ9y+JG8rOKp8IwttKzR1U95u3PSlDzvRkzfobQUq5MDAPE69FbRxvd5WwLCbAGi1HZi+Xm/7zIlzBsjsEh6dgZDgDodbu3Pb2FoHpzvewXh4sBmIFhKDAdh0A6UuBy1bUIE3CuTsSXeXtLS2i8DD3Apid2BZRuBcXIUN6N4Ho7V2vTfuNzomvamayC4lGiPUSD/rCdo+FW+n3vVtUHJgA5qSPZpiTi6SN+7wy3QX7fzzxiV/MEbXno8Yu8sVMtwo3qrTr+oy63nzzQa/zBNkRbcE3hmYJHT9EY/I83xAZ8xIyAfe0D6N3RWK5WpLV/vbvuD0acFHG0Rng2lXSFL3ERaP0+rRyQa2dZMdCpZ6yoln4cbBC2Zj1NtwsS/WFMQxbqM4FjVafE/1/g/yZUG/clkAAA==", "sha256": "d04f501f1a25cbfd13ad7e14fe088652150b07e12d8a2bee49e6225bf3a4a595"}, "run_cfpb_v052_blind_semantic_judge_v01.py": {"payload": "H4sIAAAAAAAC/9U9a3Pb1pXf+StQ9EPJlIJku04z7LKziiV3lDq2K9vZaVUNBiJBCTEIMABoW9Hyv+953Pe9AKkkOzvrDxaB+z73vM+5F3EcX26rqLvLo5uyqJbRi5dvv40+nTxPnkZtvs6qrlhEP26XtznUaert7V30ZpNXl/W2y5tkNHoPDZttVeVNVFRdDtXrKivL++gua6OqjuomW5Q5/IkWWbUsllmXH2F3a6gaZc3tFn8k0UUXNXm2bEd1BW2L9XrbZTfQrsk+R7c59J5hx+0UJ1o00abJN1mTL6NbmBJ0W90el9lNXkIFGCXKolVT/5xXo2Z70xSLJHrDs2gXdQN1o6KFKiVMpZnCIqEn+Bl9ykqcHsy07fJNMorjeDSCftZRmq623bbJ0xRmtqkbmHhV1R1PaTSS75pb6KnN5TMA4K4sbuQj/4EXyTrvMhgokyU/tnUlf9et/LWB+a3qZi2f23tV1BVrNcp2WyzV76bE/pv8p23edjx3BDjWlzOXz1Pq5ee6yrneJutwtrLaW3jkgu5+QyDj96fV/TR6ARuMuzONXhUAw6wUcNrcLxlfROVvszb/vl7mJTSpq1Vxe1Ysumn0ssjL5TRa4Z9UQX0arbGqfjHqmvvZKIJ/1Hm7aIpN1yYaG5IqX9cpQjJd5m1xC++TxWpzkwL2Pk03xSYHhM7Tdl1/zOHd03RRr9d1JWc3pr7x3/mnYplXi/zdJqum6u278+9PX7+/eJFenp++e/M6ffHm7PydUSyI4zuBzLqkrLNl2uSAa8vWeSspKpUkYFRo77Knz79OVwUAVr383ACEU8SQkl9ORvmXRb7pIgDstsxf191LpIDzpqmbWRT9PjorYOguyr/kiy1CCUkvX9/kyyWQy4saNg7pFTc/0cB9DNhwEECKfBYByOsmv6rqIyDFfHX9/xSgo1H69vLNd+cv3qeXb968T89f/xDNgRKTvPpUNIBnt3k3jl9evD67ePf2w/vz8x9OX1kt4snIfITGjFtIRGOv7wkQaFuXn/LxhGoVq8irQwXAznLRBy0iTY2mCfI/WO7Vs+vRZPTu+zd/P5eDW3M5jmJg1Ztt1x5ryjnmzUVee0xb3+b50tn/+lPeNLCN8ejyw2vZtTEOdAx4lD49efr1yZ+fPn3/5NnzZydf/wuqI3wuzk7fX7x5bcPDaZ0LUkcueqzIHlHuuK23zSJPwwNMRmfnL08/vIIVnv4XdK7mh51mn48tjkBPbd4dA7yALXZHCMn2+CbrFnfpCf5LREmsun17ef729PL8zOlbCp20qAiexH4CkGPgPj1JCMvMbt98/9bADmebFsQirW0iudsea+q00B2p80myXpoQefPhPaCoDXZvQ+gtjCi6r0GeNyTP058+59Wz9Hn67PlNmj3DQZ+kWdfl60138vx4AXt0w1OD5ZnDfg+k/AoGjLGDY+oleX4EvRxBLxoC356+O08/XFLNu67btLNjY/QkK46zTXH86Ylu8d2Hs7+dpxdnajHxwGyXeQ6yatVk6c/LJq3qqrsrqo8gvVJSbezVmNOHrfjh4uz8EuelOtGTeHf6/dtXF6//hvVeXrw6lwuFNab1alUsiqwEpGi7ZrvoUt4/2Pc3b89fX8KGnF+m/zq7BLI+e/vm4vX7d4dA4DivlpsaNKr2GNaCRPWPDxeAkqqXFDD09Ptz6Pwd9MXqDqA5A+lBsTugsgywEGAQax4Yr7MvaQdIWrXmWwQMYh5oOtbrepNu3Bcfre6Kyq4BlNIi/09hZVnZ3ZtlQESggBAOBUqRnOza7QbYQ56iKpR1VlUC9xYJUnA4UbobIU9flFnbRu860P86UkLGSh2ZsFrB+gaTHcBQqyjj/EvXZPMYxrwplrHu7e0WEGlxLrbG6xA0xh+YjYGobbc3sB9RvTL05T+00Ya6iOT2RlIZTEjdPGhaLHZxWli9ytYgigEYRuNiqd9smhqlMdCDVbHLbvVDC7rstp2hBk/PP22Ry/xMlE61ov+OXoOqCNPBP9xmu0F9AKCP+jMsI2+gh7JouytocC3GBtqoYKAlTB9fT1GBvIZuSAccg9KQbcsuXWUL4Pz3c6zGQnG7QQ0lBah36bOT9SxagbjvnGnIbZEb8rbJV2Vxe9e9q7JNe1d3Y2P7xR5tZJ0U5FtL6xNK7FUM/CHGydEPXiTLIVCsNbCaHDrNP8HCt93iIMi35fbWbP/TtmgG4dbmJehwUEWiycxBPaEfFLcFqOF6Be+bbY4LwL8KOqhP5VIjC0CE12iAQdlU8TSKhTRCqoxisrHwB1g/eaz3URElSF1S0wGuc/2WaEPOABa3IO09egmaKfUY5aJsGi3qzX2EWwxK56YE4AC4KoAhEFRs9/eHj/n9DI22bf6HCOUua7JoyLIpCAZZk90nutlkpP/vgJYYs701IDcr8+q2u5s/0ewGWaZ4+/XJyfSQFb+hZQAh3W7rbQsaOWA5kBaATk9V7jSRU1ngDwkeZ72vAeXQxm5zsj9h98ti0xZg8dbroiPju/mUI6ePPpOiHCF+be4aYFJTNAKc/n4EPAIDvTrKlj9mC7TGW9DW279E2zbXRrHcmQiQY93i8B2Y6d7k3qDVzog0R9wAcN3T2jMYJMNfi6LlTc2aAgSSuS1aDzcQ9lIw/gDCLnPozKbcbIE2EWJmk/8IAORfn4r8c3wtiA7FoElovUwIazCSyMWLVhYh7WtPHfynY+OOhThuYx7gP2nBwAPu6qVY2kr6IUD75brjBbo1/AVMoqO/6qeZgue2+ljVnyuYX0v8eYxqgWgObcJm2EQ1B3NE9DCz9rjJcAd/QHojY3O8ij+IkZSfiEeBbQaqmEUPop9drHsH5rltqtDMJMgcL8AYn+dxtoKdFv1YQJLIgAITpCNISSCpFcEmthApnplLxEqJbBvNgecLFCL/EZVKiA9DQbSTXL0l0gQF817uGG1RPBkeXGAtDQ7086gJcFtjAl1UQsuO2Cj38YuGV9j/a8a3+EcAD2Agm+5fZEGaJzNLCFdFeSan5q6Z8wt9Q3EHEMTLLTMF0MpS47Fo220u3/PD9TRSclhJQi29lTx0lRq1jEvyhrLaFlgJ47dSxmAxlhFFdW4ACErpMGpI+4kqsQmoYeKaTAy3DMQoWkCwaLR8DZ6534K5VlYX/uuzhiypqiwIqbE5XHJ+kvx5CkCEvydTwJP50+REtEUzo7/VN1arJ2arj6S4em2ennATrkg2Sn/3J+HuXUOmr4MnyXPq4Oipuyzf3OnvAyfR+ZDR1lrfQv/E0AFv16RH8zTwQxq7VBM8ufVnEF1leZMtPrYaPV5m4H0iIYc/TMXV0luDmicxaXTCLOqyZHXGwDvgB/esZtMvqg027kBfyoYFfRj9zss901S1vyzK7TIf6BnZigSqdDQpmAtXQR/UCeRPeKsmCkDgeE/RdAF7D7w1IJSWbe+GPz3RW/4LxZ9L34b0M/mQIfzyLxvWOOeGm8Cz/2cREqpTTo4ALPkmUPIRSp6eOAXsGcAmbonnIphFSER2pYCvAKsZfe3ULxD926zEVTFvBactbGBDAJmSlTyJwKCnX7ChCg47UzSKTn43V8V7pJ9VSnPey1a1pIz91sYeMKskoDP/IygTWyOwMqMI9OGClrkTynYfnsh17C4OkNGnTBgMgUsOJtCD9oNcgpF8TxE8yYuOpCsHbZF2cQc645HU2aJVVpSwZHaCjBDDwbYGD97nMeEyAHA2MqYkg1gJ1pBxrASaTBIwLthVNJ6InoCM1uAx1Y7/Mca6ZuRZRyPpHgMJrjSnYdHJILwGUFd43JP1x2XRjIX7fY6sZArYAkpeWn+kRy0LIfjZ3ANKUvPPRXeXtltAiy80g4R/R3+M4qRbb2KnWcITRkNVYxlOP1lu15t2LCYOY1ctxiazdlEUc2KDU8Bv4K3Eo1DTTsFU5plONOUAhtSo2szjbbc6+iaeOnKcJwEYU4J1SBOWAE0tlxLJmDHZ4SRpghsWxwnam+PFXYb7DYYskqJ+AnqkHhKQRnkDmw60qEphT7Oy2q7HEzmDVY7ee3TySgdJqvw6DKyveDnsRZtZOhmXsMswNVCBV97Hup8RC6PF9bqbFPa/BHSOFmXdApvdVhBvoHhz8SmPwBGsfX/ChdaSJ4DdA0KEaG+geAEzsKO7ySX/1bgx5G7W234H5jtI7vlD/AE8CUenwJQ6YKrxy6I6K1oACVi4WXksTTqK1edHCrpH6BrbmbiCaO3ODR7RqT0Wz1MJ1rkD3kkESQJ3wJlKw8oQiA0rJmzH32Ouo+JlaKOA9QKMNasIOQUpkP8Q+Uu4AgfyUDOJJ1OyyyZ6WN+w0X5be9sUI8MUBzD3iorEsWBq0i+rHPiwkCvbe5dYQj4fN/Vnlk3wA0lBTPeKp3rNtohKn6AODTI2PIIUz8V4r5wqyjk5D1PO5fZMYPJgBjKxJNpA0bIVZEeI6lU3VtnE6MzSRKm/a8t3+bi1aBgEF8NubBz9xJr6UPAkQasPPfV6MSHP9sSYu0BA5X7VvsAiA9d8C2Rj75EdklFyGmEFpKfGxeepX41XZVbkN4Gqa1gMKYNq3tBM+Fq82hwaHAgrHUUHgsVSH+xp7aynA/bz2iFIU71w/Kuva+apfXTaQvyiBQ8c81cI6lXAjtecjyLx0lWgVjHrScx0TUvnwdjdnelUtjz22usm0WMagfSdl9n6ZplF2p+vwAqBmMnVifD7C0ECvfTHNHR+gwpOzA/j/VbkYq40LMPJLbjB3GMFuo5F0fMQmZsDeqGOuQDPEOYZM/IiIXP5xpRBvopniPepAqtYD+pPwq76kZxjE1NVkZWFqqEsrSFtw9Ui9mofZF6C9+YOttBQMp7+6VAlQ/Ag0iOLltJDxp4we1k4CUJCjTsU1Rw5xZA15DSMjUl7rKO6+qQZbUGno90nSJzfBSTOkDiW09QhPMZXYLurFWCWyGTaNqiYi65jf3wLVY1J2D6Tx81Dtj1gKoo7zfWUPCQf9Ug2mO3JI6cG/FW6dNFLK9RQ2XFsqVS/hZx85PRK9DopPhFgvPHEDroC3JT1h1DW9p4Cp8XmBIOQtIYezX7rEUSeajxJOkjxLKWqCjbJcfQMQn8SYLrL/4hOUOfUL/7q0PfoMPfBKgQgSFWFzQdFafygO0yernYR/Zz8BWa8Ao30DjzsbqgzzNEoSTUFaVbWt9tcWm5g9BLfsc1gxWwMZZoaCF175joLqLC/EdqHpqbftLml6Ldc0QpXWHo8NvDGNsbnGqMw/l1CMrFcd7TeAp7d5EAR30EULKpvKAQiLUxKLuSsoQU449sxJJallt9AJoPplzogpyIZ18okfAGcEuKinElNXZLlBHKKxsKwbSBVmpNkW20OqsTnuZUVObamI3YfVuvUkouYWH2lN/dkAUQPsD1ojFzFItwSX09mZJSYxolotZODPK45NNhJ/IDYzdiexARZnPmarDl8oQZSVeDNILc524JndIFAFNPB4Rn+nMBnCAdga4GJUHhSjTs0FmDWsdoaHA6YPoTrgWLBV9euMdlQ2oW09zMXT9BeYB0QYWVMWKhLzuxmhoJEyGCXX4kOtDJNoybZBlRlxxTQUTdPlxe9zMVf3+DgcNv8IWhbqBDXGNuT7U2aJFDDZBJsYbpnx4+L2vkd7vzpqnDe3JpVrDMuU1UFgwd5wzkFMN9AZzIQOHcYqsIYtQvC7yAqeL1NwmybtgzY0avTb89fpS8vzl+dYarfowHDDI3TaFLasRTTZdrxbxwxFYqrk5HAvmBcD6W7AGRL5H+QeAK0WYr8H0Be8lnzkQpI3MasAjTEpOq51ByQ5m7ncRiUI9HHBJomFhIuyBRp0VdY+doUNfG/q1j5Iiln5t+NfrE3H+IVLYYbi+QXdvxBIPwG+D8EKik4YSZE0HIkYa5iLod61MsutjCC6tr7iUgyxv9mmpBtD6x2pxlZodgCGfSMsCzxCJwzvdB7EMAbasIVDEyOfdIRvasCs7KkHkU3qrYskcmcBgA0i4rRS7lF5g45vq3GK9iSLU6Yse0espXWkTwEJPCQxT0kLCWw27pDzEYTGUz7k9IAuiixyWcLmU8qnSt2IchJaOiQx1AkHaQBjyB4BEjFK75wFpwYVwHKTteKZbGCDKWvKQ2mDuWZobJOk5xFFWWOGf39iiQyyhozuvITyJIIHMswt7IrAFY6AYQVrJaoVJVyi39X5k788ddHOVRCmWXBy6mMpQN3ZqeaTaMQIekgkEEDQE17CMNIrWxt2tPplYfQB1QywOFQSBAqwYjPTnFITMQhRUysOpDng0o31Ep4+pQhRAmfNgMklKu2+VAzThi128lqhMPkqK9M0M48Sbs/6CqsJ5Vr1FK/iN9EC664wXGAxapZ/K7Z+WHUyWGLNVchtvvKgML1Y+PH/jKC5P1gjLHTxP6XUDx4WYsJEtUzyeeS5A8icz05n94HASlpcJ3BiZgF5LdC1vxXEIveFCml75KhGXKTEemBBcrgUycB1YE1DC9lhTyThh7g0wv2r/DpuAsqIBhjjAne0uG4Dajk62yGB1NJQQeDXxzywrNjRy3E/AsI2w/5oOMLtEbLUk7hJgeq4jOwyDZRCpCREU94pjC0KcJ4qnrfBSjm4q9mATIHTLpY5bOuIYNqokJP9omuD2cDGRME+70XcUc0kwATRExeZL2A4Vhvohs4x1vAycEqenf2d+EbaROjx1Pt+xIxfPJYbBvIhgXDBE3cglU/niUe15ATT6zkatH5/GRqcW8ddgL6Le3gLuETuS72+VtZE8C+YGmMdsoYMxi+7KZpZNaSPvsg1FuFk5Am3VDSaU7OPFcfU7al0ZtO0TXb/tFMsiOCRXn4TyBt1BE+FUj0KuQIjyUdL0GQoiZzzNsWyVSKxAtovOAJREjswuMAofXqVmIrw+BBz3Pn+pHo3El6Uy/vAyc5HsxjP7xVILOc1CJgMvT6KuRsvXayfpycNBST3Mop8JKF3PQ03dIvcxo76Wq6pVPgNMOzWaoqPBiJSdPQGSwPMCKhzZypk+vmjChy2oINuCg4h99H/4C8JDh/JNOSIpmWdIwsXR6Ui0QaWRLBuf7WDK0ZPWkggnaL3pWqU1yEjgEQWgFDRLsO2TPI3+gG7xaA9+tk5KWNiZXQo3esTBXT4/BBMgMobpmpBn38DGf122FcptCCMXggEhavIbEDXDyIZ06wN27qEvcoZsaD9rIgdwzImszIcVPolsgj7HYm1zCaXfcc4dNw1S+9Y30W8DdDR/hmOlBil4TPFWrIqXfuMT9VBZ/sg315toZiVmj7DwN61ITH0hF2qDGnIh3DoR+zaDYQlY/FuWjneHkcDNBjjje0oVywQLkczbI2BEoZ8xm7sfMwN9HcGHMC1EPAXGYBmQC3h//YdMWz1smCfNHjr75iQlCiFm2MtN7wFR5uYEAJvmIlXBwg6LVxZAyrjghCTUiRoSxM4f+PdQQ27gke9AVp+4MK4yVdK4EeoSlrlGDHTjmoOo1u6rqcDAUqZJREzlKBA4wRdFDQSwMaU+3Z9ADzqDCJcolaTh6Ztyras8sUQarmtQJGDf6VBgPRrJ0dEsQhRxyGAVRS+3Fb/Jwf4zUlSjGAOYDTCgWBUPuUdIkE49GOuL148CCYSKdIGagKkvqQOWDmKt/PAA/YeifSKBqQTTCtpXX0QG/WYGRJJxLhzQmG2RzC7MlB1vPIc8s+uDwSl4dINtaTN91icsUYsgjXUHAQN8Yk/MKonVBwPR/L0Pokucu/LItbzP9TBr7ADsUkM/A3ifwE7fBQurLQoc1zNfuUaGEZpJCKeZM3lBUvlOIQvi3u6oJ9HxKjG+VniUVhLDAbTU7hxOUSaCWqQGoMJ4Nyb7SzisEIAWyMwfWmSjbLEYTKblc2JrT1Kis4EosNt2JWRRXMpTzsjKV47fUUB1v76CajarPI99faewNV7BchIUo9BRZVLOVsTCwmkdHTQhb6zaQW1Qe7QBPDiAi0UqXmVlE+b+MwbHv/zKYW7bEuZrDU8LiBev7UobRo78S5TaMfteN2Bb+DCuIfn/J0bz/BegGIBBZyaFMbmCYq93XymL0Id2Dim9J65RoEQRsasQ8/puKZq8nQ60mPQRYSp/6YxjUaPGqwOziNCvKT4iahbkd2/LB3DNWNuy+ToesvPAU0Fh4hzArQRDWcsYcWbSrTcQKmrRIzwttjSplekRIUHMOyJuf7rM7Jm6fsbkuOKb+M9M+i+4PcSPepvhoBJfNUpRDdk6qNF2ctZdFQ1otyu3gqzg05TQYMyH0c+3FcO8C59doDTDpQKFlxoMjAjkBpkEUG6rlcK1Clh78FakpKDhTtI16HKTyKQkNt9xPaLyW2QwjOTqLW/kalA6FGZOMllAa1QK2eC2kwOjjZg2clPM12kY2y8z4MngTWgJNNNvVm7LL0kVHP3CZcWWDXRM0eNf2rr24yy41AIIFoIHbOxAlsGO9RTPC/P40t3i7Ww8mE6ITxs6YVGatM/Ri5o32RETG0VDgo8M+Y3kySlM9TpF5dqcHy/Lj21eybk5NrryoPnKKNYEhMKpvKAwRc6stM7qGn6WAbGXCwFEnZ0igMaJNia7UBFdjsK2VfXQeaSrMq2JALQ82UrRVsJ0qvQ+qybMAmMtGq75+SXOaAyj0SS0wtUGINZIkyStOw3vhKgLhQyvVoem4WR8yDSYoXqdGNFqkLi3FIHpM8RdHq5v98voN7TiBOlOmQkZKv6CEW6cSRjmUYsRPL7dAvlgXx68Pgv9AKFQnRosJA/30WKNqtPVao5n97VVuToRkuD1GLnR6UJ0X+B+mhQX/kRh+W5HsioLvfOnjGLwRJ7QuqgZeTmaI8Vj9T18heXdl61DWv/tq+7Utoah1kdeZX3l2hjiomfRDBSJ4pPemCMVfZDNTDzAlbtHEyKzCP8ZOp6euWtwZAHssTM9cZOYyb761P08jOowd7lN3xQ6DvnRfkG8uO5g8hnWM3xSsWVBn83vknn9iPbXqkxelmGWh+GI4071q/T+39nz94AYHdxHGnr8pteze3lSvzPLoge2NfvOwAp2IofGwpNkQGc/7j++4H9B6PMubWU6AvoIX5Imvz0DAST+f6Z1+SqMGfVA6Px6j6U5XfesxX3NQtuaGf+CIj3PZgwOESwbkSUSs0x16+JY7pOlzrly1BzhIEAokqZxHqoEEoAmMfb7J0ScP/7KSQyZMIOtRvJJ9oRiJST0Z20rRkOdFcZYf4mIAHhFCf+rIQWp+3LQEdATH+MJE9cU9PmOpD5KYk+Io+nHoJMT0URj1zG0BnmToyD/oVQlZKOG31kTbJgVROezWHjfCLJDTnYZtqYKfme7W8nr2Z96p6QTbhyV2ZVmYrTsp151Yfiw3xOnbdKcGFWylzmpj9W0Q03htpU/raC/nBgLl3S/h49Dj0EJeqcdL+Ut6xZteRVD6/Mi8yh3gpJboZivNkYqVUij5lcyevRd64JoeWz3Yt1tYgF2M5j8tyHYdK9QFd+WzXEooZmGi3YNCKS6lJS5tbOpvdSqc4DB3/ZRjizYr9x4i9K2R+VGrar3BM9O/pb+Wd6EPOFd8kGDFlqGBphhOGVDpII9A8fRf33jtTWhfPwHkoeYzB0V3/z+6eMaequuCZHnQvDJ01Y+8Kthk7R+P0MoPKunmkmdbUjv3AuQyOQffeETh98o225eATadCXdSSNm+tzZrj8Rx80g1NA+sYjgol9loMHkYc54J57ImXLRHOOHNpn9vwCImz3tfosgleCvcsJ+qW2EPDL/ZPw/gCU/OW9Hogvm1nB9j3IArDWuR/HOtTsER7TVWXYl0mSTA81HIP9iqWoG6P5Qj26Jnpq3RSLyn9XQ7qgqgNnc/kSp5QSt1O6/zfVLfmAZ9qSMT64OK2E+rD3l0sBiZABHQp1+D266RcDI6Miv+eWBM6iGHsXvTjYMx0NcPSJ74ugc48K7ffeRWDJPlSv9fdBxkY/E31MU7KY0EFg5wiwcZbUxVn97RFAPb4iJKdrau16WmrK+3sfTJHHN2ahe78QZwN3/Vf+ipGOuKtfebmvvKEaT5penPVc7CvhdUVT9OZKORzmaqCAJ3ktrktCjo95Ribi9XziZWxzNbotzH6lBAgnjAipIbOZ+WY/hPmhQN5WxrWJxuXFctZ4c4DZr8IG3XBAgkCOdoZ6tF4EJdaCjtECW+ETbiAiCPDq2kIBe5ywUo/pWkOeUUKRd0uEwnTGnjZJn9/SSicKPdI7rYrGGErlNK7OkK9622hFMnjrR2+7fkUW+7HoWaty++4rdmBNLjjIZMUTN8HrO8RpDkZT1m7kiXdP4/GF6sTBH65tY5HR6QGIxD0MotOlp3yg1TeIT9yrxCp5kdSCYBWas4n57hrUpEPdHH4bhkcUeFlIQLESqvRDaDTvLIOt2KRCkRwHvzU1DmlBxG1CBQ7PEZEEb/+FPd1qcKqxQ+qqqakG5r5z8MW+dCcw7OHgf0mjqfNAB6KRHMe/IovuHkENqu1cHwOsZM8FnD1bJw9O4w16+kiVb3iF9msa6lBe0kD+YNgi8+Cae1TL1lIm8ksqDUsJumEGvpsG+iCcPRTf+qryz+SIxt0/0Ucy4ejQl6mSOzlYp3TAkw4StVPudP7EZuWuNJUEOXNcARh2WMVXDzQIhBPQpKF+J7vriEgFcn07Wwzu4j2nH/d0iySLsry3U+0QkKxlbOruUgjQ88RGzAGP/YAf4GBPvSVUhtwlPe58z4E2D+N8yFsit/DKBBumFUlwaRTX3NarTGVDxODIR1di2eu5MocCrL8mjMUD7PpyEpP3X4cWtm8StvLmTEBBBUfvC1sHZqUkVHBKv/fCeXxNoLgBWryErzepCB3T7RquT6XBkP0ZvQFll/fqgoAloYd57J8+hiC99KEAt+pL8wg4GYhkRuC9cpPFrhVbZTeidZ4JmlE6mvyqBSfci888yAf6PAo87B6rSKohr5SmJl9d46SfGEazfarKNqQpb8F8YaYBb5tN3dJJHLa+QCkVX71bQuoO3C+7pjQQ/Um62E9XHsjZ4U7pIiD16RD/QBFqA5j3RTcz+UkcZh11Z5O2U3oaGCY/1DCeXD34St3TMVU3ckxDF21MgxdquOcqPScFnh9y34XPHD0qeUQcbFTHqn2wbu7Bg1rxUbjE23uuIj5/C5Xkz0T+cA9KxRtQDMVBPP9EF5/W8T+/Kwc2juD0X0bEh70R0PIjt/Qb9jxr+S1c9FF/duRmzwkucf2UBxbpWhApX+q2rqlfTeUxmU6MvgaWo0J0bjsv+hoEh9nTVDlS1EjatRKurEYZELdxj89IpseFSyf7Owku8cDuvIsbjVt6dVP/fkeTfjzEMwhq1HOz7s46Sun4kfSVv5YhaZ0gFO9R4xVHk1h3MypJLmZWUiLVGl+4Z+SNZ6kUgpT2bFqK87llq5gHH0DmWZkaOKIShObUS7jLkBZLKScoQZAZjl1tG61SVsWn0TNzusb3orbMopU0M2rJD0p6JGry9sDhUQMjHR9V6FxoCPMGm+0CnEApb3tnFHBOHDqrvU3dmTkG1965BS3twOwMZAxZb0PrCV8bY64yOIlgq30ugGAj++za0KXZYUrf1BBmuvdxskcfos87h3PV77ZAs+ltXS57VBSw7z5li3vUy+A+lLyv2g2oDnfrrPlopq8GVR48sF3CJ5LLukvpGofePo3v+zqp8YGFKJ6hLoXY07tq4Nzr0N9sZx+sVjGtR4ZkzZZ7vuzBVf9XPuyhgo84gnUfLB4E54+twC96lbwG7Qeu+FmoODS8xJiWqnDa3G4R695Sydj8FGaaLusFfLDcaJlkyyUOQ03G8dERMBVQnTAXfs4hcflhJOPz3oMdSEVkuBf5Ne89XaHmsa8j/H73YDcsN46aut7Tl/Gd7sEO5SlJtz19K26wJR56OIL7ggKN5WfkBtvzhz/I9nDbyy/MDc88+3Kk2L+ABZ3LVx/QGmyO2gyPnnFiZMyuPppO281jpe5YmC16M3FaXUEF359xvIpYgVIkjNpGJBqK6jYRlgxf4mlctH/69iL9+/k/7Su0ZQx78AYpvxP0PihONzE/UIOoxDTXJsYrI5SOX6k3Pzk4CiT0UHP+9IF/wRQV+tdLqVwjKvczjcy8PzGA8cbkOj3aNILXWOQxn4UK6M9aF97/4Z8BP6DxTYD5YAR6YvsvnOwMMzWD1w1P5vcRDOuIy+Ur6yMK0iaSVWxXpK0Gzh04+SFRglFpuEB8jc3txKrhtQ+pN24PTh2vjx4gz/eG/w1h6Y7JeIhbwpUYOab79r73mjXPfOL9kE8mQrDT+9cKauRHwCzksS+6JjFNkTulqbgpkVnV6H8AdQVMUZuJAAA=", "sha256": "788ba75d416967442a5ba82ce7fedbede2ae74608db3a6c5c052da2b7b3087bb"}, "run_cfpb_v052_blind_semantic_judge_v02.py": {"payload": "H4sIAAAAAAAC/9UbXXPctvH9fgXK9oGXnGjZbjqda9hWsc4ZpY6tke12WkXF8EjciTaPZAlS8lXRf+/u4oMASZ3sTF7qmcRHYHexWOw34CAILrqSvXh5/h27Of4mesbWRV5mTIpdUrZ5yj502VbA1DN2m7fXTNzkmShTwRqxEQ39OjuV0Wz27lqwXZWJgpXiRjQw3+5rIS1CxBCi6coSJutGSFG2kuW7Xdcm60IsWNnt1kARlq66JhWzrswBIAFeALgqboAW0OyaEkBgSbZO0o+srZj4lKStRmKt+NSytdhUjWC1aGQu27zcsvZazGQLtJImY2/11n6Ane2AC5ZWZdsAkWgWBMFstmmqHeN808FignPgsa6aFjgpqzZp86qUs5kZa7Z10khhvj/IqjS/K6ko1Ul7XeRrQ+YcPtUEyAd50+Mn5X7BXuWtaJJC81DvM3UGGuRlLopswTb4F79JijxL2qpZKLH3A7NZ2+yXMwZ/NCJInaebes3hhJ9xOmBuDpjTAcPMU5ZIOOenhEjr9yh1XgvAElzuqo8I/Iyn1W5XlWaFkLDwz0qf99s6KRd29O3qx5PX785e8IvVyds3r/mLN6ert8704EicmbbJ0/ZH3GE/KK+TZ9/8gW9y0BsanM/Ep1TULQPArhCvq/Zl1ZXZqmmqZtlvSKZNXrcy2gpQQjrKqBS7ioPYEp4JmW9ROUdb+iLxGYa+bNHo/0nWs9n5xZsfVi/e8Ys3b96xGLcduUOzi9XfT16dnZ68O4P1HaDR+Ox09fLk/StAO/mHhnFG7Oz5xer85GJ1OgAxww7cmx/PcTElI4/NJywAQ9/kW/mkP4ondHzySS/+0cE+i3ZZMJvbNd68f3f+3m5dLTTeF47Cipp8VYuyqTowbv6fW1E+59/w59+sefJ8TeebtK3Y1e3x0ycpGPFascafHbvL/vD+9PsVPzu1awYHiGZCgG/ZNAn/b9bwEvzbdV5+BG+jtddbFFeZzdIikZJdkH9+QXIK6cScgbkyJSWYPFsy2TbAzpBDS8yo53vw5KGjXJoORBCHCjm3sEaumjJugn+Hf1m+4pcnR//iV1///P1P2d3z+59P6a/57wJlX8rpL43bvAyKZC2KYMGCbYP2DxumD3XcIguuCK2oUpKwv/QuL3khym17HT9V5DGYPAYDUaUBoBzCyM/sdVUKC5uJTdIVbYyDC7YV8bHCEGX22fCE8NeBfw/xOw6SDWxaCwJwmZ4XvNpspGhlKEWxmbOjP7PAPYhgaU073zACimgTLJfE0Jz9JtbjwKod7dHo7JJcCvb3pOgEOdnQrnGEcVuJ5Qni7zqJEZmtK8gedNxnVcOSNf7S/GtufGYg3CoJJZSP9PxMT3wbO/hfxq3L57YRIMQGUoakVNtweFQJCK1jtRz9qLgQsobEQEyoeSbSXJK2WTVNUgxWqJqN+CBS/esmF7daQ4EHyCTkkhWQwFyCBl4N1YRvIGGpmn2MEFqx9J44GNYkqt3GFI2FM6tCFvAch54cA8y70qrOIQWjtKvY6xjX1XVBo4YHyt4i9h7kvoOl8rqgRJEFPkFI1IDgDpKoHDf8pz55RDnbZBMtMepR5zYSkX0MEqIw0OLTB/dXOqedaK+rbGwtGjZMC7mYkDsZkP3q1aorP5bVbQmylRCTRQYG04YaHXCmg7Cn7JrCI5q6Cd7rlWxGrlYBscFBLdmdpnM/oaZjzh4Umac8hwSXiayDs06V7DZWcJvPkZpmDGfCDOwkQu35KPYyRALzX+jwjIVxk8g7rs+zTt/3kbcwuCyOmbHL3q8YfXjEmyg82N1/uhw8HOAzDK17o0+04QlXd3hx355/KQselc9iRPskYgRd7ZdIQuE6bLSsAEzw+KXQ0vhFy3+RLA4y4RawD3p2VC/IBCk/5jaXCG1CMKXfUDu+VRhsVW5h8lpVo1Q2n+/ffnfKbq8hi1ZBsLnB2s+rXSVk8TKiEpSyD1PEOYVcvZdrZYq63jmjcdo31h8wumTst7BEst0lS5AfuAl0qEcQ427ypiqRvyNZg8A3eTrzZQi5XpvvjBTPSoh+RaHWjOPj6Hn0e1NaYxmP/NvmQDBXgQAYUGmRkoQE/0j4kRaNaMIiKbddAvlNIEoIfCkcTRm/TAop5pFGI0HPZ865XOoZiO4QmkJYDTasxyCZ6tcjlfJAr8x53ibFR25zQY7T5VaGN6g5S1V/Y61uMr4+beRZnhTVthOBDumwCOgrygcUSRFYkFp4/kUR9nR1j26XVlmo6YH+PUydwnxPDLcPibz4tGCQVOxQBAIaKMSvQplPraxaGw8IAglBdyG4Q/7uL++I/v3VyEgeZhJd+oBJcO50PioUTXEGlBCITL8qXB9t/qBbz0tHWJ+5IVrtEohf9duK7uAbY6VSinWXQxz0s5YwTSQoBFY/FD1ewGdv8G4urS2fsJYT06BFlyqhQ1GUyU6gLJDwq5PvVq/4y7PVq9O3blKBSVNSQ2GX+ZmXV0qNBKQqqXgTvOJ3uEoE6RhY2vw+WIxglbOJTak0mjfFEVAjEBkRySlSaKUApuYhE0G5RQrpEseuhki9JulIn3YNaAaI6Xg20GrHtnvFnnDJtKb9nC9Uyg7VWS9VVVDEzAeNNrBSqAkuNCd+aCK8b9nxY/FGOXfLsSpQMACTd7crQpa8VqrpmBSWHrFe6msIU6Vhaf7racX3ypSXx8+zQxrh1MwPa0UPxTEENVjGTCuGEe14PToh+v94EvYYw38P6411CkZ1AHo21J7Q8LsgXua+GlkaDzkOpSlmuFcqVXX8ivZ6+pknYxsYB07Gbnn6OPB/h6WaZxiuL3FTkeKQREoVMsiPNntlnD9qKiBQs0BpbUvfjnTGhnJqKofpa4TAC/m0nnbTAJFBmwvP5UHvDIfnuGPYyYO+XS1TJ/uiStD67izLAU7DxoOl8hZSgFbkjjYGPjmAU/JSFUvW7WpdrHygTHdCgIrWvbvTXkGgLwHiQgFBw04yCLBbSBiRFcDOqCsBeeJeYshvujW0GiJ2oYj8AFUmJLjFPnJq7OBF7gobp9l6jyX8Hh2SPmWKk1IncH4B/1P5U9mT+5ouOCLcpgy1/BZgWBIvShKZ5rlK5BZkh2UbP1tQ5OdY4sXvmk6L3kRfW8FBrs/1LY89sFCLSJVuS7/PoqQ4oQqL2ThUY3tch+r1HjccY9VstXypTsg9qodV5352oPpXvPnFCjYDcJZWns9nD1X/Byr/6Tu3celvcuXJ+wHfC2nPQkxdKjFcRWrQdxLkO3ywsSdR7nFAbOzf0bf7QJ6nn3uJo9ZNOIxpuRKwze3rpsJJdBFFobb6ldYR6EuVrcrvZzqbhF720mt1qxllWEgLKmcqAw6omSLVNFBp2lay6uhqFYQFtT+SyBfksJgdxCNfZlXCodZHGBf365gN+nFonv+EU8Oa8iavOmmFxeCzwJBBLgOdw5Nee+yF56AdtwleKB6gqKC6FW4XWAqN0K2+RNWiuOtZvQ8G5wfKAdcO6yrbL6kewAp5gcK48j2tOTFwoXf+pqpGDV+qk4rs2cqi2175KhXAcVe30MgsCrwPRoessQYTAzTdG+DQeIREFcpSB3M8N0Cmm7u0Kgolgh5zMDFAg6uYHhQ++ul7J8CoJglmYSPBiBJvyjOXUw3L9dRgRcjQii4TkwhqapqHtqr5xx6LPp1pvAWp+2n69HZQizanKyxIjJKi3bsMDOcm4qEy2QgUD/4HtlgIunWPUmrN9yZAQTc2XFAExrKjZ2UnpIROg4wvPbnc6UKTBcrisf2ubQwGPS9wv3gAEw3Tx3NN1UFzFBbv2zCPg2hpuHaGHDiQdx07wnekq25QwAdq8cXWSPxxRwjJJ97CTXIprajsiHPNC5mOmcffzkwLYt/pwO60ApSf4eCrd0kbD1QV336gmDBf4DK9ht71IHf1poaqTgB4lkiD7ma9a1mqAbHrNM51A0kXMICH6cbUvFnRyye0+jg8hXMf+X7SVHpvF/c/F16So65LIS782mFJDXB1Qf854Qr2tkmgA5mZu9/YCVct5ObicvQ4YODDdRo1HfjAweNf6o4V+q1c0PsLtqJeJfZ3x3AY6jU3XL0AopAPIUeETxfM0VgNJSELdav7Gkq2QYm1CUg4FDQMcXbnr3L/5G6C9v0oIIaGUHw3FY7uFww8uZ2D3zAypKFU0lUuyoUXfm9ejw2RsSMLYdgugdFJgGz1OBgGTGTyHi5hHEu/G5n6/XxgLZuik9exbyXzkX27ZzXqSg8Ap7Iwr49HOh+rvxZTbT5gWTuhiWLYNYPY+5qgBUofp4kUU8sY3Y37nw9Vxjo1w4aOTUPT6ypPhRz3KSc6Q0YLrZSuE0ldeUUj8Fdyk0R/scvjq0hHs0hDTfHotGQ1lOpL44U7zutB0xr/ZVswXEKBSldMg03QgzissaYcrK340BgMjz6+rXbiR+pDtdCC9Um0czHS+xx9LeKt0XsnbCB9SmfTCkKduSYEgPnl8o/Hx1c+o0W+zSHr0o+SOBwtPrUhmXAbH/UebXXobxbdAb5YQ3kN3OC3k84Pb8XMwr45g2eHLMIwoz950mVTHSjd2Iinuxzmj89R7H9+sQGTvCFMpouJhpgSTtx49f3QNrBmAcmCD+JGBrH5MUUUAa2EY/vrEWMfhUn3zcnYZEbgoZb9iLBl4DOvOJQx2kB+osgru+yV1Ll5s9gmSQJtGEb0QR/gUT3QF7+xsjZzD7yYNNnY/PCnzfWuoWG+fSiVJ8EjtQwuJ4pdMDWb2xzVfPtQOiWqGr7tgBP1jJLyo9jLlnysvnQ4VEwoYeAjHZBS16YxHgz8DTZ/6yaLozvlDzaXIsu0nmHCNKfV//FDOmB5n2nDuoN0QOM26i0FU8pu2wMJMg0vfCCF771qf8GGb2fprMKvvvp4C++lJWWbfla5dKs/XL3Hsgwa9MXMVwmwSb4pY/vhlih4N85v8Bk43lz4BYPNvUAx2gqK9hjyNCyiue2TgJglvux1sOjdjOT4dC2+tN3ixagvfOXXAKT4HNkPafvm3Xj0GlQM7v1T7VdoEGOSBThpth0qzznNhO7bLM6zKuV87mBGSZbhMoQSBkdHTXIL3GE1Fp/TrbN5Zei8sj1IAOpKmMGbh0NUzEPcR0ihBT5GCB/wHiQDSld37VFTVY/Qch7qHiSIFnVEp5iotDBQNzoBkpQt6IUyOem3WTU192j1ae+SvFTn3EcMBMDHEC40jdc5p+tv+PcCkX6jAVdPwNib89XrC9jD6oKfnJ/xv63+GdhWIV0wKszlwUccYyIYzHSHK9MEldsAFrzXv1pmYA0QYa9bpZQyUuLnKH584Izvi0FWdQVVGLfAEV1/EAE0541o02s+DRr6votp+nhZH1sgI3h6BBOzCfcAiq5wiEn48lompMLOvBlygSg8uCB+ZWG8uHRgBrIY9SgkycG9ZUc2jZM/QMkDGxHx040DZAaAI0Lj0xjIfeRMH1xK+V88GAWpNKDHfyBCaSWO9d9uPwqbqHgnq41PrWq+3H9loToAzs2U4uALL6bQeMGyOEV9+Fc/+B6FczRlzvWbFGXXs/8BP114EzM1AAA=", "sha256": "046cd0300d4aa751f3d12d02e194102d3cbccfdeb19546444aa7e8aafe65e0e4"}, "run_cfpb_v052_blind_semantic_judge_v03.py": {"payload": "H4sIAAAAAAAC/707XXPjuJHv+hVY3sNJG4nWeCbJli5MnW+tSXlvxnZsz17tOS4UTUIS1hTJEKQ9Wp//e7rxRYCkZM9d6uZhTIKNRnejvwEFQXDV5KTeMMIeecryhM0qtmIVPpFfm3TNyBOvN6Tkec5ScspYec3YA/n5A/mYxWJD5n98/y4cjW42XJBtkTYZIxVrBBMS6eP8HSmqZMNEXcU1L/KjuEl5TbJ4xyoS5ylAHA8sPSqroi6SIgsJOatJyjJ+zwADy3akeMoVcliOZTNRsoSveEJEvC0znq8lWpanZcHzepQUOSyd1EQU5K9PLJ8VOeAo4yresppVgsQVIzl7BHIEy2tSFyRu2TRowlEQBKPRqiq2hNJVUzcVo5TwbVlUNSyYF7VkT4xGZqxawyKCmfdfRZGb50KYp6bKgLOwYn9vQEQKfwps1nzLDHbzPiX4/29FzhRcGdcbmGzALuFVfah3JYpBj5/kuyn5xIHXONMclLs0zmsQmQb5yFmWTpU86WOccViyqEajutotRgT+abiqyWmyKu/p4/z3x/QehJ1SwbYSF5XKQnHDY4H7/s0Tj9XEYzlR0tlOKXkJKpAzKrbFgwSmSbHdFrlZ4Vpj+wmQbWEfp+S6rnhSf0aeRuxrwsqafJb6eV7UH4smT5dVVVSLdjWRVLysRbhmOVPKGuZsW1AQRkxTJvgaxs16YzkP/32TUKZy2uT/Z1Ej0P/douE3SL8lrLcP7Zd2PwxBo9Hl1cVPyx9v6NXFxQ2JUEKhOzS6Wv588uns9OTm7OLcBeqNj06XH0++fIJpJ/+lYZwR+/Xyanl5crU87YCYYQfu4vMlLqY488g8IgE4lRVfi6NWgEdS6OKoFVp/O8JtGowmdo2LLzeXXyzraqE+XzgKK2r0RcnyqmjAmGGfWCnAR9HHD3SFrpiiK5abE9c125b1/N1RAsZ8ryikx3N39c8Xp8tPsG5g8ByZh9njh5lEOEOEgZ3xHyfXS/rlSk7a1HUpFkcOPWHMj+KSHz06M376cvqXJT07tewFr9P/W1rRHHz2hucP4MW0bntcuWxcn3y+/HR2/hfcsY9nn5YuRx3MxQqiBI8zmmziOhhdXC7Pr2APllf0v0+v6PL89PLi7Pzm+i0sHpm4II6AXggM/0IuumElqZs4gzEZVe53hEJIwzhXUdiUjNyzrAAnvYF4hzEux7mgLU0WV1NAJ0OVDW11UdIHcNA8p+WUlBUTGCQpUBZnNbh3jHgVKyFMyJ3W4zK0xfdIQAiW9NcvZ6DillEKGn/yeQn8XwO74Bd+Y7lgtdqoZ2u2QcViCF2wFUFry8E2/kpr8AW5cEdxf9AeIDh6w0B96Q4A/SXESkZXRbWFvXA+QZ7QJIggpSDzsqnNAi+w6SCXa5axBJkkRZI0IGcZzjFF4Tm5AMav5Fb9qyBlA6qTENhbwvNHkEFR7UDSN5jqfMWEoI7XgJDLXCInSVFySHBgTwsCKRHY9Y/SyKVsgSvBRc0gSEIaYTaSoFrISCt0fgT4gCWC2QQqA0+J3GuuMiytwKHrZX4+OwUFvLw6u7g6u/mlNRSjw5r7IK5BkZOsaFI78gCavc3M6xPj600tgNp7HkP6ZcZTvuZ1nBUJi3M7FZxCfB976HBBnq+q2AzkwGRt31a8Yk9F9WDxoqqLmOP66MgTIE94ghs7Dn+iwqxKMHIwkQWBjQZuPW8kYe6BeAppURfCeB8JpFwqT7tAxuFIIJMPouGteAZr6izo9m0e4s7uBv7b53Cc2Oqo/4KssiKuAYPMrcYpW8VNVkfvwvmUrFk0x78Zi47DuZ6LNvLWWe/MrNYMF6i5vYnH8w8/yJnHv/+DmmGdkMiatRFfEDd1EcjvoK7FE13BH9CPB9HK7GOcCYYikQ8SFrNWUAvaer0W/KZqJDT+lcAyt4CEXpuvtxn5TgpbPUlo8KoHcFmXRMHR3WcsfYVMC/01yZqUHcCMAtVmuk+kUp7v1O5NrBwge6eYnoNHgMgP+UEq9u7msdzHdY37CbaDKP69k3uP8T0K4hXQGahVYDrR3xlV4Fqxx4JlqwmZ/ZkErgEGC6u8fEUQJmwNkHzXMb4WWLIUc8HIz3HWMJkmj1eBrPAqib91eFguPXt4XjS57rKOYeC6oL5QFOpPqPd68BUSvK/SBw1Xo8SYMUEzbstCra6CBH1MDoVoXdLpS9LwzYf32IO6T/Hh2VUIZsXL8eQVhgJvEoQKLCMhMSCofjtHjhUDwnK5kHW1S52BXELNnKHrv87jUmyKesDvlgaGPmIg86zvcf5eGR8+KL9ZNFXS+mCt4oAUwlpKmzpph5U+aT+8x78onCZs07KAqLxzCMCEr6x4UfF6RzEU06ZES8KqKbjzkRq4BckgHN8C9jvXE6WeK/JBoIBZc/AV1KRu4IyavPaNHC174hAMGA34QhUnMqkwsvcQ7/EqoxEaLooN0tqnsTRTIGoxcjbWVPghQpgiP4Qpk5CLQiVJ44lBZaVRxfnDGLIYKWZMC41w6qbM2K0cDMPwTq6JLNgAC8TBvFCAYdTj4CiYouOTb8D/7fwuhCDAqrEOTaYL4FBs1gohOWdfx4hTAetKu9Xy3tSM5WMz3fIEjhI2TwpXVgA2vx4PO1XJktzg/q7cLVyvDKz6TZbwSv1t/cmhSqBNTTcsBqGL6Dn4Ilg1O4Harw4WJPjI81MuIFNlS3DPbRUlc5SZNb0Z2teLW4rLvlqHNnjFemOs31XLB9iPOmKYYGm/AS8FemcpLOMdyhE4xmZTiM9jBaMW1P6KC56LOoYCYqxnTEkKHmOCTnkYADoFoCcYwoPJVMrd8W19v9bm4TL9NptJTN4PlEOBlGNejgEN+hEbKJaDiWsTAxsbekGSjaviaSLzbXjAHFsTe6sIvbvTygWrFtkj8/SK2n1RavC92hhV1y+8PFZ9UcUIxZ7bQrba1PDekP8eY/w+59WxTyeB7VUEkF2jqqsZHln7I4A2AOhYqmppauoXUyhKAYOL0YVQxh+Z3Chb2Ni2p+x6Srs2BgnEvt1a1Y5CYEvljuHkW6sNslIyL7B97RKgqMzfc1CSKNLb42Qxd0av9Rc/oH5nMlvXB0m7YmgkQ3M8x9en3YvnzI0C5l+XqXa+B+ZyiJ5YL4tMWhK9CQ5ad8J+Fz6Myo+FHks9dt7AissGeIy6Ebjs3H5HdTvUeIDwJpp77Dq0aJoSW4peLFfbcec6MRt17Vopj9d5IaCH0t+q537Gh+oD7tuui+/TgcQwXrtQ8DoApFh34dTIAOgWOJY1qWUOpgnJcD/BVR3BA32bGXmj7LwE1ifr5Vv1967j+cErodkPJOnBeaGcy76AIKAzKVZcHxYpb2STemnmnWR9FSgPpkK7m+09O/v/0k7S9ZVJ50AzusI2mtRK5YHtoize3qcxabM/n7NOEubpR9/jdyQ+a+F1qgsZfU3fz7do5lgUTgcdja99GghtXcdNGeZSqvwa6bhL6G/txk2JWwkpjOf0UAO1fHCJF50B62gCiPbXGpaatmiI3pZOeRVF1ObHLYTx/FHP7bcwHh+Ry0UL0609osMlxwByAxth5jM+tL0DhUikFe6QITsY9tQpEabN5tuky5xTp1ghuKkm5lLQVthCvH4CihnFBHHs5DRTu91azGmzLXUTAmGDiZecdXStna1TLtunOJRzdXOpV3Mw2ZyBvt8GdM1JtY4/YKol06S9irpwYwcuDJGHyt5JL5H9yDtHhGOEf7tNdLJUJWonR4e1oSOV0pp9RdedFCnEgyho6tXsB50C23S9symQBX03kAUdSsUNmW0PQPvVlK9WeE6hDlqbqsJjCoU66K/fS6yGkqdvpMNmm6+TYgNG1JLU0/zRnowEqJ1/I2kQ8kxrB9aACwEyihnEgVdO/TPym28kL8PWrPU1A+GwLaWUjwW52R4DSrntKlhxev5YTbe2ho3w/T0KyELs5Ak09OC0weT/kNEekfd/mM+NwFqUfyKyCdgO/Llj36O3NQJXQwKC/iRsPiS44+cWYXi8eiHycfJvQPEKnNiG8NrNFdxmm+/R/NM7v2jMOMMMAS9avOLBxA6y8S3igg6fatvoCkkw1WDCc3P2I7waVGANumGOvfr/Iecgdu3tYEElJbhuU6Errpk0ETxqBhtSxAo2Tmx3RlZKFqNl3Jv/u6iT6wR/y/+W/wJyQwE/8qIRbSEPr1mMeRXmbz9dX5wftbeHzM2bsJfD/ahoAOHLJgjEIWzV5mvs0iIiJY7nltSX7iaB76xiel+kUE1j70IV0yAQLKadw0uza5Di+DVAUFRq+HbIk935CVjQORWBaXpW50NnWv+ApJ3Z/9aZ3DkwaWd2PnSm4Xm0BYWX9vPLdOhQtycYfaTiUto5bemsqE9VBieoTz0aXlxLU/YTYrM+hGsl0OiQN6nCBPBAl8fPBw8lg1smBJi7iDrFX1AVGZIXKPODUjnQyo6Zr2uSL9M9M9FC/HmuzTjTHM1xzxQ01c6QAyfPGQwEvjgs2QM+y7gdcZLAGkS1jeSRl5uLeqfsUWef4ZaYZA3zE6o7cJ2ddT8tDhTRgbz14l14wbs/eGsjGCyX8YAC5mGDfOi7WREdmXSIV5oVve0OXeNuNTuo7627iNrH9hYSOnh1pAzO9Z/t29UAFZsYDmHf4vOBtxUcrEMk1ieRkePzVSOwf9vNd4K6AzgcPcBD4h8JIQtPJvv0ZClb93i1og+HbQFNDc2b7T2eAUIZAD6bjeEs1NFNc3pKfkfeOXkN1Ep53Y3cbYllkJNnf5WXo+cB3C+9iGKLsuh5yJ+/TPFE2X6DZxjp4lBq6SpXJBXUXk+lENuEHrM+TltdD5tp3D87/n7gkPhFTF0jf+5Z+cukY0KrrBGbyDedSc/o3c3rHeJ0AIdyG/efMoJI/enbqyJZ+6f+Z88uIu9tABdYQZTEgg0tY5Q5ah+ne45mnfzcMAoBpuAJE4se5n7OfWlv+LgnFnlBNI7AX6lNvbqL4VmaDkuhhhqi0Tlx0VBTNFh7IqMHh8+V38qCoRKS4+4Bs6pM4bZyqtPHIa/rV7Qam4/DWIrGYmdA046aYxgDM1YLTkmbnjoHiK0zAsHDmM9y67ZgJfg6GlYULBfragwAk9vFD/P5nU9s2wVHL0xhi/HmoZQNtcFT8zk2Az7D6CfwTjHKreMf/zToFeXddL2wb9bg8vF+hCZGv1J5Xb5vknDCmMrWGMgu1C99k/EpivzXbzZkKW+In0n/kxFOZB4GkctqgGKtuKNGBlG//ep2CHfUSjiyT68YfS9+osajFaFD7JtOD3ysZd9DbAlYDHLH84YNXPiwEf5EoVf22SrpRPU9UIvtbJNBgTZ0Q72vC6/rgQpUIlLWpnPzTq1hTDIyD/7nFG6h4q0Rg8O8+1AqgYI7u2kUZNk2GPraNnPNuw+lc6WiouuGp/rOuUycIi+N8me1tcCh6kAJ45GzJ91zxo0ZaDv3rt38apMsaZnWMwyY5rD6v75JByzvjTasexcHNG4VyDGilN0W3jESDdfSIL9vvSpe4tKH5/BDA7lX4++/f3iCH7UImYb66aZ3mQVXb2dZAs306chXCbBJusoj++LWL9goM1eWInlByUnpTVJmfikU4TEcNs5tBwLELDot/RVe8xEUr0RHtwFuCABhYWfRNTmHO8dOGWeDWL+PjYS/0ur2iwxpQBTFoC4CmR8JheegqqKME+2f5CDGNgtwUq0bVMJL+QXuEqrfcKBgKE2LhNKJMzOM0xSXkVPGwWxWxU/AJZZ8kWy4E3MZ0fmRxEEEwBB8YelhLOZ3FK+gQkt+DRH+/uIgGnV6MauK4hVczu8sXqNL9YpaFOr6wMFZaM8zqUOxSk6DuITTzlRiEYBCG7zwr7dobK5CaB3ZxjxX2tHGKwQAbfCg5XjJKZxWwqdChCx/5BXc+5F3dZxDuJPLM/qfy1/8rrWeuTh0ljuABEOp6TtrhOYIiDqHI2+5cuP4Pe/auH+kh8zaWs7x0s7ZlYLRA6gOKFUMnkeyGzZAQCiPtNwDMuVrkPK+5wLbcdaBN/eEUFmF890MeceI6LFdEL/4MQFGDLODfPR6K0LykDmuDck08ecAJg+sh8TPhA6g6QD2EA04y//7XjlRYS9hKpDgNirI7vw9oVbbQ6T/DhywGjtWq5o3V4lUj0PevcPjUzFWFEADIRcylxcJ56ZfgBcnc3mdHM+HcU3VWpigHwAjpTJ9gd+YwoWagFL0CpTqm0zKRYz+AfyqUEO9OwAA", "sha256": "22beadb343cd8ecdae297b35d1d9c3f8437941fcb365bacf8699699cfa295d36"}, "validate_cfpb_v052_pipeline_smoke_v02.py": {"payload": "H4sIAAAAAAAC/+1cW3fbRpJ+56/AYM5MAJukLk6yGe5oHUWWs8raslaSM3tC8eBAYJNCDAIcXCQrov77ftU3dAMgJCfztGf1IAmN6urq6urqujVc1/2wWCRxypzbMInnYZnlzu3uvrPA36O3Zz84F4zN0fLNeN9Zx2tGoKNilX1izjwOk2xZsWI8GJyHd846z27jOcudrCrXVenEhROvVlUZXids7DiXN2ioB7nL45IVTpk5YeqwzyXL0zAZ5ExCxFmKAXIWAfZ+6FQFYM/u52FaxhEnbslSlocliCuiG7YKiyEQzZ0yZ2FZDAq0cNBfq/lyxdKycMICw6yTOIpLhxGhacQcYLgBxeUNEZFG2TxOl3hiKydOnes8C+cgaVklYU6dc1YUIKzAbI5vWX6vZppjtDgtwIH4NozuR1WKt/EiBnGgaXANzDerMP80AvOSeBkTQwau6w4GizxbOUGwqMoqZ0EAhq2zvESvNCs5E4rBQLXly3WYF0w934TFTRJfq8dfiyxV/2eF+i9nYogoSxLwkhCqMY6yKgXXxXtwnJXxiqmX6nno0O/fslTiWYNfGFSBneFRvCjv18Q62X6YYs3eYYFzrKnsqRZPgvwQFux9NmfJEJSki3j5Jo7KofM2Zsl86PyspeA4z7N86CyoPdDiMxiU+f1k4OCHYy+iPF6XxVhKBfqNU7bKAgCHwZwV8RLt42ixvg4gy/uBEuWAizLa9oMoW60gdJI8j+Omnw95GCXsPLsb6qYLKV0/SeGq39AqBEW4YHVTAiEKMo4kiIrbxgtIeJbPi0arEt9Ai28NUNyE+998GyzixBiE76aARk9Eo/+lvIFwQ7xAIvGowKbfxqg2h94cvz38+O4yeH9yGvx4/uHj6ZuT0x+Dyw//dXx6UZO4zCFwtL+CBbYoxL1QlLLPEVuXDqShSthpVr4lQL7uE8f5s/OGqwHsPxZVXC8IgSN1Iqb2VVGrinE97y9ZbRoHIswmDriR5WyaZqOczdli9n9HDr58hZ/BlT+69oNBcHb+4afjo8vg/MOHy+D49GfnAApszNLbOIeoLlnpuW9PgPfi7OPl8fHPh++sHq4/MB/RWQgmqSavhdsfY+wsuWWeYEy8cFow/AVLCiZxcBYHgdF1THzEYkxfzQb+4OI9ZqsGt2jZcVxxQhQ79ebbEXzGCt3vbFuKDOdHjiPKHZx/PFWojXGAOK/SYH93/9vdf9vfv9x79c2r3W9/ATjx5+TN4eXJh1ObH43exjlb7Gi1Squ/U2RVHrGgewB/oFb8/PAfQK7pI6Th3Y6lVPhTwcod8OufFStHxMli5zoso5tgl37G8o2r0Z6cYpkbiKXkzoM45czk6murBO/vjvkGaOAM3h+enrw9vngKeQMbbw2wDeMFK0qOuUZMQvnh4oS4XQueIQG8AWMoPkQVN1l2jPPdloEgrOYxuCH71Wuw9+py7+tvvt5/9cuOtDKCf4Y7iypJBA+wcLqXnoGCnMfFOitiGg+Ae3IS9VJKmTl+Q6xpSRAwSgEB5bXR1+Dx+THNejuKnP0KDd2P4eeT439s738bs7u+3mcfzi+fmAAxAMsN5UYMk3wY/Bm2KcN2h3FDZihMxRyGEIyjlDqESXLvpCGOoztnzgCwitO44JYMQJZQuPcOLNDoE5mFp8fnQEeGqLZAIywy4GED3jvXLMnIwMycMyiTeB5nO8U6PIIhmTuhE4HIa2HTkr6HjXh2chKcHV5eHp+fXmBqD3yFXbI2E3cC426MY2yNPeXl7tX19HD0y+7ob+PgLy9Hs5ffq0f8fzWmh9nD/vDx6todUscTX2hgd30D485GptV27nqv//6nq7nvvZ5cvXy9Nx2Nr4rZa/81PXuvr+YPrx6v/NeqmT/LB/z/9aP3mjoLuVTjFUXaJF0OwruP8Huf/9bdVc8q75j0TVmui9eTnZ2ri5ebq+u7u7urMf5tTJK4jl2wjEtSa9vH/9vQGPRxcIFDDJv7Z+jN4//+KFRHk09EBNghnZ9NwdL5BkdzzjYlSxJnxTaMzOxNREZuvvKvrscPu8NX3zy6VvewKO5gAGxgRW+i21vgga7gogUbeYMlmo6cGbfQeQP2AABp+zthFJElv1H4eGMUAleRRdgtjkK1Aet9Wn8BCPYMoQPenpxfXAZnx+cX2C6HR1KVbZnlyeaOkTB8lSQb5y6m31GYbuCL3LKNE66cZcYdqMy/Kl4oggBPen9TVNeruNxka5ZuYnhI2ENLCPsGdliMs2ezyBn7jW2ukyz6tAHaiCUb7HoGnwd/FzAg9Bwj6GugYgW2DKGQJzMxuQyjsj3L4/85e3dydAI1cfLjf15eBEfvDk/e2/OkKd5nlQNy+XRyVsKp88kbdPJ4eVNiWvXuGbw5OSctD23//uTieDvLJNc2QHNvsc7fYLSchpOTE5PyYdhlVTKXgvJ1Q1AsWIM9av6rdQIdtYG6io3nuWQgUz1ZzUuJY66QzNvcO/zh4sM7GF/ds5WULaswh8ZjbIO9sIpBFQxFqEpsg/tNxHLiZnLvGywcXFyefzy6/HgOk+78+BACSEoOtupvLMVpKdj4oLWRi9UlUoUaF14/jmeu2t3hFrCEpcvyJgA9K7I6TLgVeZ+BjiMESQj1HHBH0wSr0k9pdpfCdCcsBRmx/LTWgz4OyIyNEmxh5z0iBOGSedq79YWTKsbiOmCJOdYer4fQRx4euAhpXAOjsErzLIHNLf3nqYvgRw6muRgABwl47Ar7m6QdymXiFGUOpNx19nA6yUkf7NV0vZHH5h8kzGTtxElAzVTOWFBU3KeQc3JYIowTFNUKYY/7PgKplzJTYBOxQqJFj9mAv/2+4fp71gJLwr7n01yx8iab8wbIngo2scDs4EUJQkUrQXXRmITvjP7DbploQYCrALI91dN3QC5FiR6+HjrfDp3vHmtQvoYhtgDFMSrGXVmbbGdVFSVfQOwKR2CADfCdJkzOiztveBFj737WZNOwLK1WXHBrimwCEK/iBhcYLySIZsDxOH9x9p2DA2dXeDmGXFkIAC5Rj0kgnT8daJz2SM+YLiEoxKTDhMf6Sia8Uk5aPVlo3SpPNRsgwLSSPPAVQCaqiLzHuXdLA00ozsSXbA6BJZEZUotcMw6CyWtXXHTSfl+MrUyzjuSLIcdiMFGSwl9u74RRjT46KqV+OOVzScaYnPrCpEMsFA9/cICfoATfMDriRfzDFilB0MNjk0Q5iE2faJSzEiste9Z4BHfJ8AjSZR6uCq/Etuf7dehgh8PCBenfcR5DI0/LCseJYPR4PJ5JTlP/QpwJUPhzGMw4E6bh6DeYn1/NyBYjrJj8Hcs937doqDlHqD2OaiqkdCKl9aWTzjp2A20CHDhQtqvws7c75LuT9we5Topue3IsNU+tD2DKyzM6u5s0hEfodK7hr++h5M33NqQGbcRoJq24kLNxTmHCgUX0Bycq8VPwUiu7JnYuWZLBCBWfC34hug0xgQLTkfURIu446ueYS1XyP3ciEs6fxzzKLBjOexrqFeRMZ/VsY5JStHvAIKIu+pgDAZKV1ERgmj0cUgLqrcWhEJ6j2Rr7SRB4wCdKo5i7nhNHFPWdt7Om4Gt2CNzD2s2UsyZpQey7WqUkLh6cfQQZo5KmBHM0MB4xVMVUu3jwLeVfoyFPkLt4BrPEW8Ep0pT0jshuvWxuaT6BcbiGVTz3FoLn5EywlHZxEN2QiM8nDwLHI1SlWAitC8G0lnrUS6jcZbV+wHwTX8fa3Cm0Vyn8JG4CGVwRDQ1WiUbNsBpGNtX4vojdypqS/G6T+lfSQV49TUtXW3zste0EJyxNrdgEbihLaSxQKJ1hDjswtHYjUbGdoj7j1e8ihDbPoE2qOoID7vUBDjq6Xu8O21ctvSTYu0RAVyZV6uPa7xlgtKeWRFMn9zaZLKR1VfvYHN43DQaB7JncadrsUuTr1aTjhMwaYa9o86XR3iCYzDVbIbURXqXu+Fe4sJ61SafKDJIGN9cqhinWOf2ZheKlM9VQ3UZyE16DW9axBqpFpj397lk0JmG9e86Eem1DWJNdZqR9bKchJRRllI3GMcNbY+x02B8N/Uh6QMCPCxbm0Y1nL5rfYYk2VSoSuOwzgsTx5IEoeDS2G9DDaJGYyY+llPfVNcURCjKvYc6QFr4aL7NbKCt7bOnG9ip0l0d34XyznPLN9tDyWFOGEXHEHkG96R9CosEWWyPjbA/hTonHR4dIn1CAd+a2B+lHDrZBTJG74YbTOgkjdpMlsDk6h2sFzBRvbSF9YkI1MKJpFLa+xTlIeSfYVGECbiJjUJT2yB1hrD84dpWGt4izko0VhDx1HsDJjFf2uJ2Bpd81csKWmBysHh5sKroGs2NOW0YhpdwM2PxBVsC4QrCHEXFzFs5lboEHebQ9IpJGwouAEjJ9CssWcutMWFCnBelwytelNDXFpCkJloRrMjct5H+1kDf0gdL8svP2E0fi/I6w1LIsjEaZo8TI7cTl750NaFIopm4NmqKjuatK5JzSwp05f+9NrW6fF/y/arFAlQt2S02SmhrXOg1/pftwVHipNAd4W73GEsAf2AZ8geQKI3hYJRJCjo2g+DxYYEuBBwC8zrLEs5BsI46bGmRwE5UmloZvQWefSBO5Mof71HStTE09tTmLYir04ceaxPicsawZYhvWB/C/gArya2QqpZMW5feo7HUHkE7jKZ+w2yMTzVOy3ml3wAgDWcIGd2eWwyCcksl2b8c6/3+/I2Z4BoK4cbXmVrnG/2CpMreRxKVkO7I+l3ll1EooD+WWczlMtgEI80tVdAHqbQj2NsB0jVegKry2AK6ra9SgCY3xBOgqvA948iggOyQJZBZ5CzSSvGUeX1fipJKpxP5RHo0SEb3ojXKQOnFKKWZ7+V3eidJp9Nfw++S25ok24aHX79oqFWB2WYhIFJoKWqpz4UYAnrwO2eabfZqbKKCaPca79G7BPhzoa1vS7a0qfMV5tVp79O+ByzPLftOU7SXBDt5SuM5q9S23+imZfFIerUhgK4zC/8qoGVWKVGT8hUsqcixlDZMn4wmMdAezwmQUKaQAkwyQCXgTQJdSSQiNJeADFxYyilR1xc+6Yr6Idr3HXEj9ijmNEmS3EpFOd9Yw6LN8RRY9ws6UqCyi+FNcAiiEg6FDZZa/LUroPgkQJBjyONIllEi9oqADC1GgeItcm1yqbCrnQhg0WPAXcPHWBN+Nk+pP8ixiPNSlML+vkjJ+Ryr3B2z9PP4NhZqGE3/CobjnLupaI14pt87D5SqcQKSgayEciIPKMioStlEBXxzyEg3soP05thQSyyJsb+uUI5k/HIk1VLLAI/qwwmMykkwe/rvDw89IQFPWlErJJBivXRsbjqHgAQgfmBFIw4AQI8q0EyV3mPHSOFdLlGNVKSV2XnCjQ/SbSoSzsY5AUNWfMjf4caSinjiP1PDGgbVp4NYyakY8hwLoKXzavKG9LifTtLQCJVsHzbNsFUd5FmjZwj7eG+8Ou2CE1PUALPaaLx877AVeCM0oY9chhp6k/0D+9duBIxgMefwZ3SUiZAXgcCBWX9D2a8RWnrNe29g76wg01LqkRcY2EqRQ9a6w/4U0iLWiEAEtCeqX94ZOQKHafhXhdabw5FSGrckh0o9tjijMgVhgmE1YIpT2xsKoOdjtoO7L5W2BvFXp6Ra/X/gEtHj0t4uhAFvs+S1h/Ax3m/Z6ZSxU//Y7OHCevfctk7RjQcWORdEY2Rh6I8vjssCGaGaMmibRXSENFN3ZLE4QuwYQ8r/63YsX9soYvThLEA0pA3XkAAFvbAXN7ZO027erUlXrQRmXhm3DaW5FB7cxyrButnJfOzIqY+5iFkz4x9KvqVOaMzPgURP6RHK9FQVcuG/UqErQ5VHkiMF3xMiSasrpwOx4qAd8tJPhHWpOTatoMZFC8k9zw3Dr/C/i96xD2fUT0xQLK2tnEGLF876MFEVAYCi/ua3rjAalDu2F+yLl2L0WwxZMB4/aQFqNXtM5d++2IRp6dbhFOkw1S8pi6hpMIBYbipU0h9CCbeY9jUfq2w4kUvU+iQFquKP3Yq+np1Y/dUeLEbZF7HWsT9d6NNmn3RIavZm8R7BESLEY+YUsVQvvAlFrP+Fl+9qxMGq7zTd2Wbf5xiyWputO5jsdQGm9Ufun4wUFiTqaeRVys7l94cMAabpBtXPVA7TVV1pniEPc25UpjWJ3nEcirOqpu3EHblUuRt+5dVBToBHB0HWVg3WI11Dqz+2/ZyJCMoPt9UMnRIujaFEHWdh9F1GBSfVFausTwx00XQQVm7UmzwoP9YaG/hUuOAfiDilS7nGS4XBPKOHTwPXYzow12c7/x2n68OjzFkzd316u1WL1oslrKRoqAYtzERhV3kwPPpWXW9zZ1BWXkaAVqPShvpjk2RvQ71vvMwkqtia/3ugYKeCueffdc5AU+U2Kmtu7nyY5gipRNJVDJ4Xme2tnNYd91t4igfeM2gCNojH/WooDKiSnU5z2C4lsnSZGjLmFASte8Yya0FEiPqK6cxns758zuP3IYUfk/fN6FXP8ZyDo2CONrk/ITHtZ5hkTqmJNVzXKWlMQfxzOKpUNoUPDsITrM6R9S6y+PlW/U/asZ+IZOi/MR3VVrHcadHtaXiXmJZLiPrFDBds5j6RwqsUEkPv6hDsjOpdMCpaMPvMWYXPbqck2weoJ+42qN/JpZHnTVNeBzfwJr30i7Up/pUFYaO9DOz0cB996qqn4kr0v62JxrbNa82A500Yor51U1bFEvwgaTXsK10x60WNmU1t6Eouv6VUNDjd+awCuSpThLKb4zFXlBXE8JEcVFvN4gXqAwrqVKaeuJLO+tm7ayFvuhHpbrIZGsUEnTGcE2iwWFUhUUZ5Jl5Hp081Uetlgkd5CLRw9zFu4F+0b/KpcWiISXD15I/y2Bm5dKCdC2lLMrcgyL4RslHQ2nPk6YTZBraRZXqifRXJQVVI+GtlQkazAkPKmvSq6lI+e/4wIOpHz+LwYuYa0xX0yaLj2dZWnubObed9GLgCdrOpZXmVaL/KwXiXLtfQbxZ9qGab8caaS1uJlK0Il+KdSfa34jeaIdrIdsT+3Q0remR1kqkMxjvbEkGxHrtg1Bs+2/IemaBjhHc9yA0x5sYG0S1CLkAQwNIpxl9sTsA0mgsqZlHLpBtQxPb2fdWCmtd3rkWQM5KB5R90zeg5V/CRQl4oOyL5o6BgV9+jXld36UiUbFDnP1Jf2JAwW9CauWqIxlAiGLWk5aDYY3qrhyz0RFpS3TjEPEVN1KZ067FA1AYpt8F59gWMMveapj3CM8c4fx0XG3QaEPk2PRDpfQJ2JT7oE5i3vIFv03/x327lXun6sC83bPhCdiyQalOY1TR5YhSEv2iozr8tu8schDh/Ya5+tCRAOabVPLJPdRG6nQU0rp7tvwxIyetu+bndvG8Ya+xm+x6Tf8WhM3Qgc49HvFA0TxtYFhj6aWZiVQurpqpVUoyfXUr39uObqGE8EmVQNAK8ukdaCrdqlZ+v35/ANEmpDo6uH0hhaRbV6Uayzg8O27kAvu8GEFJMHyMOjKQ7CBZ/8fx3K76lDeWYkhBtYBtOpIrpVnOGirKDABVvcBKwoXNT6EhS/3S8rn3ARXwgxbEpcUMPN/8YVf7dBo6ztChNy3+9l1K/gX4waO0eqTladYFA+2PoJP3Tpli1ddptTwh53CRACcJvFHbX9pkulbxp2iLHPh41wpD90vHovD+2QpHzJN+zQDEr6LS+bn1RTLemzKREza+cJpeqX0Qz1aZTfp/ybqbOGquEUNMG79b2tWR+bwVbpj49Xn/DJIE8659ySoWBhTFbCJ8OwMXsKe4yHbKzv/Iyp7gfONAcFlpTqT4IQdRHxgZBefmEtLQ/2h9xpCmC4iTENSptRIPPzPToezqtIzCuR+CgY9CeFeNXnwcan4BUPEMgwLzVSLl8DHObLijbCGX+DwL/4ihKlNYJgnkX41I3RcxzO5zQM7+K5oxFU50gG/ob860AHPHxNN25DhOsPjO/D9OJRJ/NIGHK9uPgHXHqxcSQjHQZ+Gpv+HMwTRIovuhmHdz9u44swvYjNbdyDTn+ZpX9J6k3ftyDyCy1PoJIqohcRfajlCTQkqk+hoS+29KJRynukT3ATZW9XoYLb4OaV2Xxs7iK5sSj0JrZU7SQRgL5nJ6Fr9WA6x43slJ2aOqCu4/p5aHqohpkq4Oy2GtY2SgWs3TbsihlzJaYwt2zWuot9qIgOum1oHBPG6SInJptMIH3OKBBqGDYOGwuAa9HthdOFAdz3eTTDdRXA0r0z1CrYAFn5oyqc5IY+IxbQIYVPOFI2PQhIioJAJriESA3+F5H6FH1tUwAA", "sha256": "2e9b9f47f51bc3571366883311f1f9200f839335c06aaa7c6e16bf7c602c24de"}}')
SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)
for name, item in embedded.items():
    raw = gzip.decompress(base64.b64decode(item["payload"]))
    if hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise ValueError(f"Embedded file hash mismatch: {name}")
    target = SNAPSHOT_ROOT / name
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != item["sha256"]:
        raise ValueError(f"Existing frozen source snapshot differs: {target}")
    target.write_bytes(raw)

if str(SNAPSHOT_ROOT) not in sys.path:
    sys.path.insert(0, str(SNAPSHOT_ROOT))
for module_name in (
    "cfpb_v052_pipeline_smoke_v02_common",
    "prepare_cfpb_seed_v052_pipeline_smoke_v02",
    "validate_cfpb_v052_pipeline_smoke_v02",
    "run_cfpb_v052_blind_semantic_judge_v01",
    "run_cfpb_v052_blind_semantic_judge_v02",
    "run_cfpb_v052_blind_semantic_judge_v03",
    "evaluate_cfpb_v052_semantic_judge_v01",
):
    sys.modules.pop(module_name, None)
from cfpb_v052_pipeline_smoke_v02_common import sha256_file, load_semantic_judgments
from run_cfpb_v052_blind_semantic_judge_v01 import load_blind_cases
from run_cfpb_v052_blind_semantic_judge_v02 import build_evidence_units
from run_cfpb_v052_blind_semantic_judge_v03 import (
    RunnerConfig,
    resolve_zdr_endpoint_preflight,
    run_judge,
)
from validate_cfpb_v052_pipeline_smoke_v02 import validate_and_route
from evaluate_cfpb_v052_semantic_judge_v01 import evaluate_judge

PROMPT = SNAPSHOT_ROOT / "cfpb_v052_semantic_judge_v02.md"
RUNNER_SOURCE = SNAPSHOT_ROOT / "run_cfpb_v052_blind_semantic_judge_v03.py"
COMMON_SOURCE = SNAPSHOT_ROOT / "cfpb_v052_pipeline_smoke_v02_common.py"
VALIDATOR_SOURCE = SNAPSHOT_ROOT / "validate_cfpb_v052_pipeline_smoke_v02.py"
print({name: item["sha256"] for name, item in embedded.items()})


## 3. Inspect payload, verify the live ZDR endpoint, and provide the API key


In [ ]:
from getpass import getpass
import pandas as pd

cases = load_blind_cases(RAW, PREPARED)
assert len(cases) == 20
assert all(
    set(case.model_dump()) == {"seed_id", "labels", "grounding", "generated"}
    for case in cases
)
preview_units = build_evidence_units(cases[0])
print({
    "rows": len(cases),
    "fields_visible_to_judge": ["case_id", "evidence_units"],
    "evidence_protocol": "stable_reference_ids_v01",
    "oracle_loaded_by_runner": False,
    "candidate_judgments_loaded_by_runner": False,
    "prompt_sha256": sha256_file(PROMPT),
    "first_case_evidence_units": len(preview_units),
})
display(pd.DataFrame([
    {
        "ref_id": unit.ref_id,
        "source": unit.source,
        "location": unit.location,
        "text_preview": unit.text[:180],
    }
    for unit in preview_units[:10]
]))

CONFIG_TEMPLATE = RunnerConfig(
    model_name="deepseek/deepseek-v4-flash-0731",
    base_url="https://openrouter.ai/api/v1",
    judge_id=(
        "openrouter_deepseek_v4_flash_0731_zdr_"
        "nonthinking_blind_v02_attempt01"
    ),
    sampling_profile="deepseek_v4_flash_0731_official_chat",
    temperature=1.0,
    top_p=1.0,
    max_tokens=2048,
    provider_slug="auto",
    allow_fallbacks=False,
    require_parameters=True,
    data_collection="deny",
    zdr=True,
    reasoning_enabled=False,
    reasoning_exclude=True,
    max_attempts=2,
    request_timeout_seconds=120.0,
)
CONFIG, endpoint_preflight = resolve_zdr_endpoint_preflight(
    config=CONFIG_TEMPLATE,
    output_path=ENDPOINT_PREFLIGHT,
)
print(CONFIG.model_dump())
print({
    "zdr_endpoint_preflight": "passed",
    "model": endpoint_preflight.model_id,
    "provider": endpoint_preflight.selected_endpoint.provider_name,
    "tag": endpoint_preflight.selected_endpoint.tag,
    "status": endpoint_preflight.selected_endpoint.status,
    "eligible_endpoint_count": endpoint_preflight.eligible_endpoint_count,
    "supported_parameters": endpoint_preflight.selected_endpoint.supported_parameters,
})

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass(
        "OPENROUTER_API_KEY (not persisted): "
    )
if not os.environ["OPENROUTER_API_KEY"].strip():
    raise RuntimeError("OPENROUTER_API_KEY is empty")


## 4. Two-row DeepSeek API/schema contract smoke

Run this before the full calibration. It proves routing, non-thinking
response content, strict JSON schema, and evidence-reference resolution.
A second call is allowed only after a non-empty response fails the
JSON/schema/reference contract; provider errors and empty content stop.


In [ ]:
CONTRACT_IDS = [
    "cfpb_v052_0d210ece132c66d3",
    "cfpb_v052_2c70d920e4cfccd5",
]
contract_report = run_judge(
    raw_path=RAW,
    prepared_path=PREPARED,
    prompt_path=PROMPT,
    judgments_path=CONTRACT_ROOT / "semantic_judgments.jsonl",
    raw_responses_path=CONTRACT_ROOT / "raw_responses.jsonl",
    failed_attempts_path=CONTRACT_ROOT / "failed_attempts.jsonl",
    endpoint_preflight_path=ENDPOINT_PREFLIGHT,
    report_path=CONTRACT_ROOT / "judge_run_report.json",
    config=CONFIG,
    api_key=os.environ["OPENROUTER_API_KEY"],
    selected_seed_ids=CONTRACT_IDS,
)
assert contract_report["semantic_coverage_complete"] is True
contract = load_semantic_judgments(CONTRACT_ROOT / "semantic_judgments.jsonl")
display(pd.DataFrame([
    {"seed_id": key, "decision": value.decision, "reasons": value.reasons}
    for key, value in sorted(contract.items())
]))


## 5. Run/resume the blind 20-row development calibration


In [ ]:
import shutil

# Reuse the two contract-smoke judgments only when starting a fresh
# calibration. They used the exact same prompt, config, and schema;
# their separate originals remain as contract evidence.
calibration_judgments = CALIBRATION_ROOT / "semantic_judgments.jsonl"
calibration_responses = CALIBRATION_ROOT / "raw_responses.jsonl"
calibration_failures = CALIBRATION_ROOT / "failed_attempts.jsonl"
if not calibration_judgments.exists() and not calibration_responses.exists():
    CALIBRATION_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CONTRACT_ROOT / "semantic_judgments.jsonl", calibration_judgments)
    shutil.copy2(CONTRACT_ROOT / "raw_responses.jsonl", calibration_responses)
    contract_failures = CONTRACT_ROOT / "failed_attempts.jsonl"
    if contract_failures.exists():
        shutil.copy2(contract_failures, calibration_failures)
    print("Bootstrapped the fresh calibration cache with 2 contract-smoke rows")
elif calibration_judgments.exists() != calibration_responses.exists():
    raise ValueError("Calibration judgment/audit cache is incomplete")

calibration_report = run_judge(
    raw_path=RAW,
    prepared_path=PREPARED,
    prompt_path=PROMPT,
    judgments_path=calibration_judgments,
    raw_responses_path=calibration_responses,
    failed_attempts_path=calibration_failures,
    endpoint_preflight_path=ENDPOINT_PREFLIGHT,
    report_path=CALIBRATION_ROOT / "judge_run_report.json",
    config=CONFIG,
    api_key=os.environ["OPENROUTER_API_KEY"],
)
assert calibration_report["semantic_coverage_complete"] is True
print({
    "rows": calibration_report["judgment_rows"],
    "new_provider_calls": calibration_report["new_provider_calls"],
    "decision_counts": calibration_report["decision_counts"],
    "failed_attempt_rows": calibration_report["outputs"]["failed_attempts"]["rows"],
})


## 6. Score the fixed model ablation

This is the first stage that loads the versioned GPT-5.6-SOL development
oracle v02. It measures development agreement, not human accuracy. The
same conservative rubric allows direct comparison with Qwen attempt01.


In [ ]:
EVALUATED = CALIBRATION_ROOT / "evaluated"
evaluation = validate_and_route(
    raw_output=RAW,
    prepared_input=PREPARED,
    input_manifest=INPUT_MANIFEST,
    disposition_path=PRIVACY_DISPOSITION,
    validated_path=EVALUATED / "validated/dialogues.jsonl",
    rejected_path=EVALUATED / "rejected/dialogues.jsonl",
    review_path=EVALUATED / "review/dialogues.jsonl",
    report_path=EVALUATED / "validation_report_v02.json",
    semantic_judgments_path=CALIBRATION_ROOT / "semantic_judgments.jsonl",
    oracle_path=ORACLE,
)
metrics = evaluation["oracle_metrics"]
calibration_metrics = evaluate_judge(
    judgments_path=CALIBRATION_ROOT / "semantic_judgments.jsonl",
    oracle_path=ORACLE,
    combined_report_path=EVALUATED / "validation_report_v02.json",
    output_path=CALIBRATION_ROOT / "calibration_metrics_v02.json",
)
display(pd.DataFrame({
    "judge_only": pd.Series(calibration_metrics["judge_only_metrics"]),
    "combined_validator": pd.Series(calibration_metrics["combined_validator_metrics"]),
}))
print({
    "validated": evaluation["validated_rows"],
    "rejected": evaluation["rejected_rows"],
    "review": evaluation["review_rows"],
    "development_only": True,
    "human_gold": False,
})


# Direct model-only comparison when the synced Qwen attempt01 metrics exist.
if QWEN_ATTEMPT01_METRICS.is_file():
    qwen_metrics = json.loads(
        QWEN_ATTEMPT01_METRICS.read_text(encoding="utf-8")
    )["judge_only_metrics"]
    deepseek_metrics = calibration_metrics["judge_only_metrics"]
    comparison_names = [
        "decision_accuracy",
        "decision_reject_precision",
        "decision_reject_recall",
        "decision_reject_f1",
        "exact_set_accuracy",
        "micro_precision",
        "micro_recall",
        "micro_f1",
    ]
    display(pd.DataFrame([
        {
            "metric": name,
            "qwen_attempt01": float(qwen_metrics[name]),
            "deepseek_attempt01": float(deepseek_metrics[name]),
            "delta": float(deepseek_metrics[name]) - float(qwen_metrics[name]),
        }
        for name in comparison_names
    ]))
else:
    print({"qwen_comparison_skipped_missing": str(QWEN_ATTEMPT01_METRICS)})


## 7. Preserve results; do not freeze from this notebook

This development set has informed model and rubric choices, and its
oracle is LLM-produced rather than human gold. Even excellent metrics
are evidence for selecting the next experiment, not freeze approval.


In [ ]:
ZERO_RECALL_CODES_TO_INSPECT = [
    "claim_type_changed",
    "factual_status_changed",
    "unsupported_scenario_detail",
    "indirect_sensitive_information_guidance",
]
print({
    "frozen": False,
    "freeze_supported_by_this_notebook": False,
    "next_review_focus": ZERO_RECALL_CODES_TO_INSPECT,
    "development_only": True,
    "human_gold": False,
    "privacy_verified": False,
    "benchmark_eligible": False,
})


## Stop point

Preserve the complete DeepSeek attempt as independent lineage and sync
its output directory back to the local project. Compare it with Qwen
attempt01 before deciding whether to run the attempt02 high-recall rubric
on DeepSeek. Do not edit this notebook in place, do not freeze from these
20 development rows, and do not start a formal pilot while the privacy
release gate remains closed.
